# 📈 Proyeksi SARIMA — Google Colab

Notebook ini menjalankan model **SARIMA** untuk proyeksi deret waktu tanpa perlu menginstal apa pun.
Mesin perhitungannya **sama** dengan *ARIMA Forecasting Platform* (aplikasi web).

**Cara memakai:** jalankan sel **satu per satu dari atas ke bawah** dengan tombol ▶ di kiri setiap sel
(atau Shift+Enter). Isi pengaturan di kotak formulir sebelah kanan sel, lalu jalankan ulang sel tersebut.

| Langkah | Isi |
|---|---|
| 1 | Menyiapkan mesin SARIMA |
| 2 | Mengunggah data |
| 3 | Memilih kolom & memeriksa data |
| 4 | Pengaturan model |
| 5 | Menjalankan SARIMA |
| 6 | Diagnostik residual |
| 7 | Grafik & tabel hasil |
| 8 | Evaluasi periode uji |
| 9 | Perbandingan dengan EViews *(opsional)* |
| 10 | Mengunduh hasil |
| + | Uji otomatis & tampilan web *(opsional)* |

Bawaan model: **log(data), SARIMA(0,0,0)(1,0,0)12 + C**, data latih 2021-01 s.d. 2025-06, uji 2025-07 s.d. 2026-06.
Konstanta **C** diestimasi sebagai rata-rata seperti di EViews.

> 🔒 Data yang diunggah hanya ada di sesi Colab Anda dan hilang saat sesi ditutup.

In [ ]:
#@title Langkah 1 — Siapkan mesin SARIMA (cukup dijalankan, tidak perlu diubah)
import os, sys, importlib
from pathlib import Path

FOLDER = Path("/content/sarima_app")
FOLDER.mkdir(parents=True, exist_ok=True)
os.chdir(FOLDER)
if str(FOLDER) not in sys.path:
    sys.path.insert(0, str(FOLDER))

BERKAS_MESIN = {
    'modules/__init__.py': '',
    'modules/data_loader.py': '"""Membaca file CSV/Excel dan meringkas isinya. Tidak mengubah data."""\nfrom __future__ import annotations\n\nimport pandas as pd\n\n\nclass GalatData(Exception):\n    """Galat yang pesannya aman ditampilkan langsung ke pengguna."""\n\n\ndef baca_file(file) -> pd.DataFrame:\n    """Baca CSV atau Excel. `file` boleh path atau objek unggahan Streamlit."""\n    nama = getattr(file, "name", str(file)).lower()\n    try:\n        if nama.endswith((".xlsx", ".xls")):\n            df = pd.read_excel(file)\n        elif nama.endswith(".csv"):\n            df = pd.read_csv(file, sep=None, engine="python")  # deteksi koma / titik koma\n        else:\n            raise GalatData("Format file tidak didukung. Gunakan .csv atau .xlsx.")\n    except GalatData:\n        raise\n    except Exception as e:\n        raise GalatData(f"File tidak bisa dibaca ({e}). Pastikan file tidak rusak dan tidak sedang dikunci.") from e\n\n    if df.empty or df.shape[1] < 2:\n        raise GalatData("File harus berisi minimal dua kolom (tanggal dan nilai) dan tidak boleh kosong.")\n    df.columns = [str(c).strip() for c in df.columns]\n    return df\n\n\ndef ringkasan_dataset(df: pd.DataFrame) -> dict:\n    """Ringkasan untuk halaman unggah: jumlah baris/kolom, tipe data, nilai kosong."""\n    info = pd.DataFrame({\n        "Tipe data": df.dtypes.astype(str),\n        "Nilai kosong": df.isna().sum(),\n        "Contoh nilai": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],\n    })\n    info["Contoh nilai"] = info["Contoh nilai"].astype(str)\n    return {"jumlah_baris": len(df), "jumlah_kolom": df.shape[1], "info_kolom": info}\n\n\ndef tebak_kolom(df: pd.DataFrame) -> tuple[str, str]:\n    """Tebakan awal kolom tanggal dan kolom target (pengguna tetap bisa mengganti)."""\n    kolom = list(df.columns)\n    kata_tanggal = ("tanggal", "periode", "date", "bulan", "waktu", "time", "tahun")\n    tgl = next((c for c in kolom if any(k in c.lower() for k in kata_tanggal)), None)\n    if tgl is None:\n        tgl = next((c for c in kolom if pd.api.types.is_datetime64_any_dtype(df[c])), kolom[0])\n    angka = [c for c in kolom if c != tgl and pd.api.types.is_numeric_dtype(df[c])]\n    target = angka[0] if angka else next(c for c in kolom if c != tgl)\n    return tgl, target\n',
    'modules/data_preprocessing.py': '"""\nMembentuk deret waktu dari tabel, memeriksa masalah data, dan transformasi log.\n\nAturan: data tidak pernah dihapus atau diubah diam-diam. Setiap periode yang\nhilang, nilai kosong, atau nilai yang tidak bisa dibaca dilaporkan ke pengguna.\n"""\nfrom __future__ import annotations\n\nimport re\nimport warnings\nfrom dataclasses import dataclass, field\n\nimport numpy as np\nimport pandas as pd\n\n# nama tampilan -> (kode periode pandas, panjang musim bawaan)\nFREKUENSI = {\n    "Bulanan": ("M", 12),\n    "Triwulanan": ("Q", 4),\n    "Mingguan": ("W", 52),\n    "Harian": ("D", 7),\n}\n\nTRANSFORMASI = {\n    "log": "Terapkan log natural (data diunggah dalam nilai asli)",\n    "sudah_log": "Data sudah dalam bentuk log (jangan di-log lagi)",\n    "tanpa": "Tanpa transformasi",\n}\n\n_POLA_BULANAN = re.compile(r"^\\s*(\\d{4})\\s*[Mm]\\s*(\\d{1,2})\\s*$")   # 2021M01\n_POLA_TRIWULAN = re.compile(r"^\\s*(\\d{4})\\s*[Qq]\\s*([1-4])\\s*$")   # 2021Q1\n\n\ndef ke_periode(kolom: pd.Series, kode_frek: str) -> pd.PeriodIndex:\n    """Ubah kolom tanggal ke PeriodIndex. Mendukung format EViews (2021M01, 2021Q1)."""\n    teks = kolom.astype(str).str.strip()\n    if kode_frek == "M" and teks.str.match(_POLA_BULANAN).all():\n        b = teks.str.extract(_POLA_BULANAN)\n        return pd.PeriodIndex(b[0] + "-" + b[1].str.zfill(2), freq="M")\n    if kode_frek == "Q" and teks.str.match(_POLA_TRIWULAN).all():\n        b = teks.str.extract(_POLA_TRIWULAN)\n        return pd.PeriodIndex(b[0] + "Q" + b[1], freq="Q")\n    if pd.api.types.is_datetime64_any_dtype(kolom):\n        tanggal = pd.DatetimeIndex(kolom)\n    else:\n        # Teks tanggal: format Indonesia hari/bulan/tahun (01/02/2021 = 1 Februari). ISO (2021-02-01) tetap benar.\n        with warnings.catch_warnings():\n            warnings.simplefilter("ignore", UserWarning)\n            tanggal = pd.DatetimeIndex(pd.to_datetime(kolom, errors="raise", dayfirst=True))\n    return tanggal.to_period(kode_frek)\n\n\ndef label_periode(p: pd.Period) -> str:\n    """Label ringkas untuk tampilan: 2021-01, 2021Q1, atau tanggal awal minggu/hari."""\n    kode = p.freqstr[0]\n    if kode == "M":\n        return p.strftime("%Y-%m")\n    if kode == "Q":\n        return f"{p.year}Q{p.quarter}"\n    return p.start_time.strftime("%Y-%m-%d")\n\n\ndef ke_angka(kolom: pd.Series) -> pd.Series:\n    """Ubah teks ke angka. Mengenali format Indonesia (1.234.567,89)."""\n    if pd.api.types.is_numeric_dtype(kolom):\n        return kolom.astype(float)\n    teks = kolom.astype(str).str.strip().replace({"": np.nan, "nan": np.nan, "None": np.nan})\n    format_id = teks.str.contains(r",\\d+$", na=False) | teks.str.contains(r"\\.\\d{3}(?:\\.|$)", na=False)\n    teks = teks.where(~format_id, teks.str.replace(".", "", regex=False).str.replace(",", ".", regex=False))\n    return pd.to_numeric(teks, errors="coerce")\n\n\n@dataclass\nclass HasilValidasi:\n    seri: pd.Series\n    galat: list[str] = field(default_factory=list)\n    peringatan: list[str] = field(default_factory=list)\n\n    @property\n    def ok(self) -> bool:\n        return not self.galat\n\n\ndef siapkan_seri(df: pd.DataFrame, kolom_tanggal: str, kolom_target: str,\n                 frekuensi: str = "Bulanan") -> HasilValidasi:\n    """Bentuk deret waktu reguler dan laporkan semua masalah data."""\n    kode, _ = FREKUENSI[frekuensi]\n    galat, peringatan = [], []\n    if kolom_tanggal == kolom_target:\n        return HasilValidasi(pd.Series(dtype=float), ["Kolom tanggal dan kolom target tidak boleh sama."])\n\n    tanggal_kosong = df[kolom_tanggal].isna()\n    if tanggal_kosong.any():\n        peringatan.append(f"{int(tanggal_kosong.sum())} baris tanpa tanggal tidak diikutkan.")\n    df = df.loc[~tanggal_kosong]\n\n    try:\n        periode = ke_periode(df[kolom_tanggal], kode)\n    except Exception:\n        contoh = ", ".join(df[kolom_tanggal].astype(str).head(3))\n        return HasilValidasi(pd.Series(dtype=float), [\n            f"Kolom tanggal \'{kolom_tanggal}\' tidak bisa dibaca sebagai tanggal (contoh isi: {contoh}). "\n            "Gunakan format seperti 2021-01-31, 31/01/2021, 2021M01, atau 2021Q1."])\n\n    asli = df[kolom_target]\n    nilai = ke_angka(asli)\n    tak_terbaca = nilai.isna() & asli.notna()\n    if len(nilai) and tak_terbaca.mean() > 0.5:\n        return HasilValidasi(pd.Series(dtype=float), [\n            f"Kolom \'{kolom_target}\' sebagian besar bukan angka. Pilih kolom target yang berisi nilai numerik."])\n    if tak_terbaca.any():\n        peringatan.append(f"{int(tak_terbaca.sum())} nilai di kolom target bukan angka dan dianggap kosong.")\n\n    seri = pd.Series(nilai.to_numpy(), index=periode, name=kolom_target).sort_index()\n\n    if seri.index.duplicated().any():\n        dup = sorted({label_periode(p) for p in seri.index[seri.index.duplicated()]})\n        galat.append(\n            f"Ada {len(dup)} periode ganda untuk frekuensi {frekuensi.lower()} (misalnya {\', \'.join(dup[:5])}). "\n            "Periksa apakah frekuensi yang dipilih sudah benar, atau agregasikan data terlebih dahulu.")\n        return HasilValidasi(seri, galat, peringatan)\n\n    lengkap = pd.period_range(seri.index.min(), seri.index.max(), freq=seri.index.freq)\n    hilang = lengkap.difference(seri.index)\n    if len(hilang):\n        peringatan.append(\n            f"{len(hilang)} periode tidak ada di data ({\', \'.join(label_periode(p) for p in hilang[:6])}"\n            f"{\' ...\' if len(hilang) > 6 else \'\'}). Periode ini dibiarkan kosong; model tidak mengisinya dengan nilai karangan.")\n    seri = seri.reindex(lengkap)\n\n    kosong = seri.isna() & ~seri.index.isin(hilang)\n    if kosong.any() and not tak_terbaca.any():\n        peringatan.append(f"{int(kosong.sum())} nilai target kosong.")\n\n    if seri.notna().sum() < 10:\n        galat.append("Data terlalu sedikit (kurang dari 10 observasi terisi) untuk model ARIMA.")\n    return HasilValidasi(seri, galat, peringatan)\n\n\n# --------------------------------------------------------------------------- transformasi\n\ndef cek_transformasi(seri: pd.Series, mode: str) -> list[str]:\n    """Galat yang mencegah transformasi dijalankan."""\n    if mode == "log":\n        buruk = seri.dropna() <= 0\n        if buruk.any():\n            contoh = ", ".join(label_periode(p) for p in buruk[buruk].index[:6])\n            return [f"Transformasi log membutuhkan nilai > 0. Ada {int(buruk.sum())} nilai ≤ 0 (misalnya {contoh}). "\n                    "Pilih \'Tanpa transformasi\' atau perbaiki datanya."]\n    return []\n\n\ndef petunjuk_log_ganda(seri: pd.Series, mode: str) -> str | None:\n    """Peringatan bila data tampaknya sudah dalam log tetapi akan di-log lagi."""\n    s = seri.dropna()\n    if mode == "log" and len(s) and s.min() > 0 and s.max() < 30 and (s.max() / s.min()) < 3:\n        return ("Nilai data kecil dan rentangnya sempit (seperti data yang sudah di-log). Jika kolom ini sudah "\n                "berupa log, pilih \'Data sudah dalam bentuk log\' agar tidak di-log dua kali.")\n    return None\n\n\ndef ke_skala_model(seri: pd.Series, mode: str) -> pd.Series:\n    return np.log(seri) if mode == "log" else seri.astype(float)\n\n\ndef ke_skala_asli(x, mode: str):\n    """Kembalikan dari skala model ke skala asli data."""\n    return np.exp(x) if mode in ("log", "sudah_log") else x\n\n\ndef potong(seri: pd.Series, awal: pd.Period, akhir: pd.Period) -> pd.Series:\n    return seri.loc[awal:akhir]\n',
    'modules/exploration.py': '"""Ringkasan eksplorasi data (hanya informasi; tidak memengaruhi model)."""\nfrom __future__ import annotations\n\nimport pandas as pd\n\nfrom .data_preprocessing import label_periode\n\n\ndef statistik_deskriptif(seri: pd.Series) -> pd.Series:\n    s = seri.dropna()\n    return pd.Series({\n        "Jumlah periode": len(seri),\n        "Observasi terisi": len(s),\n        "Nilai kosong": int(seri.isna().sum()),\n        "Rata-rata": s.mean(),\n        "Median": s.median(),\n        "Simpangan baku": s.std(),\n        "Minimum": s.min(),\n        "Maksimum": s.max(),\n        "Periode awal": label_periode(seri.index.min()),\n        "Periode akhir": label_periode(seri.index.max()),\n    })\n\n\ndef daftar_kosong(seri: pd.Series) -> list[str]:\n    return [label_periode(p) for p in seri.index[seri.isna()]]\n\n\ndef dekomposisi(seri: pd.Series, s: int, model: str = "additive"):\n    """Dekomposisi klasik (moving average). Butuh data tanpa kosong dan ≥ 2 musim penuh."""\n    from statsmodels.tsa.seasonal import seasonal_decompose\n\n    if seri.isna().any():\n        raise ValueError("Dekomposisi tidak bisa dijalankan karena ada nilai kosong.")\n    if len(seri) < 2 * s:\n        raise ValueError(f"Dekomposisi butuh minimal {2 * s} observasi (2 musim penuh).")\n    if model == "multiplicative" and (seri <= 0).any():\n        raise ValueError("Dekomposisi multiplikatif butuh semua nilai > 0.")\n    hasil = seasonal_decompose(seri.to_timestamp(), model=model, period=s)\n    return pd.DataFrame({"Data": hasil.observed, "Tren": hasil.trend, "Musiman": hasil.seasonal,\n                         "Residual": hasil.resid})\n',
    'modules/arima_model.py': '"""\nEstimasi (S)ARIMA dan diagnostiknya — dipindahkan dari reference/arima_engine.py\ntanpa mengubah metodologi:\n\n- Konstanta C diestimasi sebagai RATA-RATA (regresi pada konstanta dengan galat\n  ARMA), seperti EViews; bukan intercept `trend=\'c\'` statsmodels.\n- Maximum Likelihood, kovarians koefisien outer product of gradients (OPG).\n- Stasioneritas/invertibilitas tidak dipaksa secara default (seperti EViews).\n- Kriteria informasi juga dilaporkan per observasi, seperti EViews.\n\nstatsmodels diimpor di dalam fungsi agar modul lain tetap bisa diuji tanpa statsmodels.\n"""\nfrom __future__ import annotations\n\nimport re\nfrom dataclasses import dataclass\n\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats\n\n\n@dataclass\nclass SpesifikasiModel:\n    order: tuple[int, int, int] = (0, 0, 0)\n    seasonal_order: tuple[int, int, int, int] = (1, 0, 0, 12)\n    konstanta: bool = True\n    paksa_stasioner: bool = False\n    paksa_invertible: bool = False\n\n    def label(self) -> str:\n        p, d, q = self.order\n        P, D, Q, s = self.seasonal_order\n        if P == D == Q == 0:\n            teks = f"ARIMA({p},{d},{q})"\n        else:\n            teks = f"SARIMA({p},{d},{q})({P},{D},{Q}){s}"\n        return teks + (" + C" if self.konstanta else "")\n\n    def jumlah_parameter_arma(self) -> int:\n        p, _, q = self.order\n        P, _, Q, _ = self.seasonal_order\n        return p + q + P + Q\n\n    def minimal_observasi(self) -> int:\n        """Batas bawah kasar agar model bisa diestimasi."""\n        p, d, q = self.order\n        P, D, Q, s = self.seasonal_order\n        lag_maks = max(p + d, (P + D) * s if (P or D) else 0, q, Q * s if Q else 0)\n        return max(10, lag_maks + d + D * s + self.jumlah_parameter_arma() + 5)\n\n\ndef exog_konstanta(index, pakai: bool):\n    if not pakai:\n        return None\n    return pd.DataFrame({"C": np.ones(len(index))}, index=index)\n\n\ndef estimasi(y_model: pd.Series, spek: SpesifikasiModel):\n    """Estimasi model pada data skala model (mis. log). Mengembalikan hasil statsmodels."""\n    from statsmodels.tsa.statespace.sarimax import SARIMAX\n\n    model = SARIMAX(\n        y_model,\n        exog=exog_konstanta(y_model.index, spek.konstanta),\n        order=spek.order,\n        seasonal_order=spek.seasonal_order,\n        trend="n",\n        enforce_stationarity=spek.paksa_stasioner,\n        enforce_invertibility=spek.paksa_invertible,\n    )\n    return model.fit(disp=False, maxiter=2000, cov_type="opg")\n\n\ndef nama_parameter(nama: str) -> str:\n    """Nama parameter statsmodels -> gaya EViews (ar.S.L12 -> SAR(12))."""\n    if nama == "sigma2":\n        return "SIGMASQ"\n    m = re.match(r"^(ar|ma)\\.(S\\.)?L(\\d+)$", nama)\n    if m:\n        jenis, musiman, lag = m.groups()\n        return f"{\'S\' if musiman else \'\'}{jenis.upper()}({lag})"\n    return nama\n\n\ndef tabel_koefisien(hasil) -> pd.DataFrame:\n    tabel = pd.DataFrame({\n        "Koefisien": hasil.params,\n        "Std. Error": hasil.bse,\n        "t/z-Statistic": hasil.tvalues,\n        "Prob.": hasil.pvalues,\n    })\n    tabel.index = [nama_parameter(n) for n in tabel.index]\n    tabel.index.name = "Variabel"\n    return tabel\n\n\ndef residual_efektif(hasil) -> pd.Series:\n    """Residual tanpa periode awal (burn-in) inisialisasi difus, bila ada."""\n    burn = int(getattr(hasil, "loglikelihood_burn", 0) or 0)\n    return pd.Series(hasil.resid).iloc[burn:].dropna()\n\n\ndef statistik_ringkas(hasil, y_model: pd.Series) -> pd.Series:\n    n = int(hasil.nobs)\n    k = len(hasil.params)  # EViews ikut menghitung SIGMASQ\n    ll = float(hasil.llf)\n    resid = residual_efektif(hasil)\n    y_sejajar = y_model.reindex(resid.index)\n    ssr = float((resid ** 2).sum())\n    sst = float(((y_sejajar - y_sejajar.mean()) ** 2).sum())\n    return pd.Series({\n        "Jumlah observasi": n,\n        "Log likelihood": ll,\n        "AIC (statsmodels, total)": float(hasil.aic),\n        "BIC (statsmodels, total)": float(hasil.bic),\n        "Akaike info criterion (per obs., EViews)": (-2 * ll + 2 * k) / n,\n        "Schwarz criterion (per obs., EViews)": (-2 * ll + k * np.log(n)) / n,\n        "Hannan-Quinn criter. (per obs., EViews)": (-2 * ll + 2 * k * np.log(np.log(n))) / n,\n        "R-squared (perkiraan)": 1 - ssr / sst if sst > 0 else np.nan,\n        "Sum squared resid": ssr,\n        "Observasi burn-in (tidak dihitung di residual)": int(getattr(hasil, "loglikelihood_burn", 0) or 0),\n    })\n\n\ndef diagnostik(hasil, spek: SpesifikasiModel) -> dict:\n    from statsmodels.stats.diagnostic import acorr_ljungbox\n    from statsmodels.tsa.stattools import acf\n\n    resid = residual_efektif(hasil)\n    s = spek.seasonal_order[3]\n    df_model = spek.jumlah_parameter_arma()\n    lag_uji = sorted({lag for lag in (s, 2 * s, 12, 24) if df_model < lag < len(resid)})\n    ljung_box = acorr_ljungbox(resid, lags=lag_uji, model_df=df_model) if lag_uji else pd.DataFrame()\n\n    jb = stats.jarque_bera(resid)\n    nlags = max(1, min(2 * s, 24, len(resid) - 1))\n    nilai_acf = acf(resid, nlags=nlags, fft=False, missing="drop")[1:]\n\n    akar_ar = 1 / np.abs(hasil.arroots) if len(hasil.arroots) else np.array([])\n    akar_ma = 1 / np.abs(hasil.maroots) if len(hasil.maroots) else np.array([])\n\n    return {\n        "ljung_box": ljung_box,\n        "jarque_bera": {"statistik": float(jb.statistic), "p_value": float(jb.pvalue)},\n        "acf": pd.DataFrame({"lag": np.arange(1, nlags + 1), "acf": nilai_acf}),\n        "batas_acf": 1.96 / np.sqrt(len(resid)),\n        "akar_ar_invers_maks": float(akar_ar.max()) if len(akar_ar) else None,\n        "akar_ma_invers_maks": float(akar_ma.max()) if len(akar_ma) else None,\n        "peringatan": peringatan_model(hasil.params, hasil.bse, akar_ar, akar_ma, s),\n    }\n\n\ndef peringatan_model(params: pd.Series, bse: pd.Series, akar_ar, akar_ma, s: int,\n                     ambang: float = 0.99) -> list[str]:\n    """Deteksi model yang tidak bisa ditafsirkan seperti ARIMA biasa."""\n    pesan = []\n    if len(akar_ar) and np.max(akar_ar) >= ambang:\n        pesan.append(f"Akar AR invers = {np.max(akar_ar):.4f} (≥ {ambang}): komponen AR di/dekat unit root, "\n                     "model tidak stasioner.")\n    if len(akar_ma) and np.max(akar_ma) >= ambang:\n        pesan.append(f"Akar MA invers = {np.max(akar_ma):.4f} (≥ {ambang}): komponen MA tidak/nyaris tidak invertible.")\n    for ar, ma in ((f"ar.S.L{s}", f"ma.S.L{s}"), ("ar.L1", "ma.L1")):\n        if ar in params.index and ma in params.index:\n            phi, theta = params[ar], params[ma]\n            if abs(phi + theta) < 0.05 and abs(phi) > 0.5:\n                pesan.append(f"{nama_parameter(ar)} = {phi:.4f} dan {nama_parameter(ma)} = {theta:.4f} hampir saling "\n                             "meniadakan (pembatalan akar); t-statistik yang sangat besar tidak bermakna.")\n    if (bse.drop("sigma2", errors="ignore") < 1e-3).any():\n        pesan.append("Ada standard error sangat kecil (< 0,001): estimasi berada di batas ruang parameter.")\n    return pesan\n\n\ndef ringkasan_teks(hasil, spek: SpesifikasiModel, stat: pd.Series) -> str:\n    """Ringkasan model sebagai teks (untuk diunduh)."""\n    garis = [f"Model: {spek.label()}", ""]\n    garis.append(tabel_koefisien(hasil).to_string(float_format=lambda v: f"{v:.6f}"))\n    garis += ["", stat.to_string(float_format=lambda v: f"{v:.6f}"), "", "--- statsmodels summary ---",\n              str(hasil.summary())]\n    return "\\n".join(garis)\n',
    'modules/forecasting.py': '"""\nBacktest dan proyeksi. Metodologi sama dengan kode acuan:\n- pembagian latih/uji kronologis (tidak diacak);\n- model diestimasi HANYA dengan data latih, lalu diramal dinamis multi-langkah\n  sepanjang periode uji (tidak memakai nilai aktual periode uji);\n- proyeksi ke depan: model diestimasi ulang dengan data s.d. periode terakhir;\n- hasil dikembalikan ke skala asli dengan exp() bila model memakai log.\n"""\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom . import arima_model as am\nfrom .data_preprocessing import ke_skala_asli\n\n\ndef proyeksi(hasil, langkah: int, spek: am.SpesifikasiModel, mode_transformasi: str,\n             alpha: float = 0.05, koreksi_bias: bool = False) -> pd.DataFrame:\n    """Ramalan dinamis `langkah` periode setelah akhir data estimasi, dalam skala asli."""\n    akhir = hasil.fittedvalues.index[-1]\n    index_depan = pd.period_range(akhir + 1, periods=langkah, freq=akhir.freq)\n    ramalan = hasil.get_forecast(steps=langkah, exog=am.exog_konstanta(index_depan, spek.konstanta))\n    rata = np.asarray(ramalan.predicted_mean)\n    se = np.asarray(ramalan.se_mean)\n    ci = np.asarray(ramalan.conf_int(alpha=alpha))\n    pakai_exp = mode_transformasi in ("log", "sudah_log")\n    faktor = np.exp(se ** 2 / 2) if (koreksi_bias and pakai_exp) else 1.0\n    return pd.DataFrame({\n        "ramalan_skala_model": rata,\n        "se_skala_model": se,\n        "ramalan": ke_skala_asli(rata, mode_transformasi) * faktor,\n        "batas_bawah": ke_skala_asli(ci[:, 0], mode_transformasi),\n        "batas_atas": ke_skala_asli(ci[:, 1], mode_transformasi),\n    }, index=index_depan)\n\n\ndef baseline_naif_musiman(y_model_latih: pd.Series, periode_uji: pd.PeriodIndex, s: int) -> pd.Series:\n    """Ramalan = nilai periode yang sama satu musim sebelumnya (berulang bila horizon > s)."""\n    nilai = y_model_latih.copy()\n    for p in periode_uji:\n        nilai.loc[p] = nilai.get(p - s, np.nan)\n    return nilai.loc[periode_uji]\n\n\ndef baseline_rata_musiman(y_model_latih: pd.Series, periode_uji: pd.PeriodIndex, s: int) -> pd.Series:\n    """Ramalan = rata-rata (di skala model) per posisi musim pada data latih."""\n    y = y_model_latih.dropna()\n    posisi = pd.Series([p.ordinal % s for p in y.index], index=y.index)\n    rata = y.groupby(posisi).mean()\n    return pd.Series([rata.get(p.ordinal % s, np.nan) for p in periode_uji], index=periode_uji)\n\n\ndef backtest(y_model: pd.Series, spek: am.SpesifikasiModel, awal_latih, akhir_latih, akhir_uji,\n             mode_transformasi: str, alpha: float = 0.05, koreksi_bias: bool = False) -> dict:\n    s = spek.seasonal_order[3]\n    y_latih = y_model.loc[awal_latih:akhir_latih]\n    y_uji = y_model.loc[akhir_latih + 1:akhir_uji]\n    hasil = am.estimasi(y_latih, spek)\n    ramal = proyeksi(hasil, len(y_uji), spek, mode_transformasi, alpha, koreksi_bias)\n    aktual = ke_skala_asli(y_uji, mode_transformasi)\n    tabel = pd.DataFrame({\n        "aktual": aktual,\n        "ramalan": ramal["ramalan"],\n        "galat": aktual - ramal["ramalan"],\n        "batas_bawah": ramal["batas_bawah"],\n        "batas_atas": ramal["batas_atas"],\n    })\n    pembanding = {\n        "Naif musiman": ke_skala_asli(baseline_naif_musiman(y_latih, y_uji.index, s), mode_transformasi),\n        "Rata-rata musiman": ke_skala_asli(baseline_rata_musiman(y_latih, y_uji.index, s), mode_transformasi),\n    }\n    return {"hasil": hasil, "y_latih": y_latih, "tabel": tabel, "pembanding": pembanding}\n',
    'modules/evaluation.py': '"""\nUkuran akurasi ramalan — definisi sama dengan kode acuan (reference/arima_engine.py):\n\nMAE   = rata-rata |aktual − ramalan|\nRMSE  = akar rata-rata (aktual − ramalan)²\nMAPE  = rata-rata |aktual − ramalan| / |aktual| × 100\nsMAPE = rata-rata |aktual − ramalan| / ((|aktual| + |ramalan|)/2) × 100\nTheil U2 = akar[ Σ((ramalan_t+1 − aktual_t+1)/aktual_t)² / Σ((aktual_t+1 − aktual_t)/aktual_t)² ],\n           dihitung di dalam periode uji.\n\nSatu-satunya tambahan: periode dengan aktual = 0 dikecualikan dari MAPE (pembagian\ndengan nol) dan jumlahnya dilaporkan, bukan disembunyikan.\n"""\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nPENJELASAN = {\n    "MAE": "Rata-rata selisih absolut antara nilai aktual dan ramalan, dalam satuan data. Makin kecil makin baik.",\n    "RMSE": "Seperti MAE tetapi memberi bobot lebih besar pada kesalahan besar. Satuan sama dengan data.",\n    "MAPE (%)": "Rata-rata kesalahan dalam persen dari nilai aktual. Mudah dibaca, tetapi membengkak bila ada nilai aktual yang sangat kecil.",\n    "sMAPE (%)": "Versi simetris MAPE (dibagi rata-rata aktual dan ramalan), lebih stabil saat nilai aktual kecil.",\n    "Theil U2": "Perbandingan dengan ramalan naif \'sama dengan periode sebelumnya\'. < 1 berarti lebih baik dari naif.",\n}\n\n\ndef ukuran_akurasi(aktual, ramalan) -> dict:\n    a = np.asarray(aktual, dtype=float)\n    f = np.asarray(ramalan, dtype=float)\n    sah = ~(np.isnan(a) | np.isnan(f))\n    a, f = a[sah], f[sah]\n    if len(a) == 0:\n        return {"n": 0}\n    e = a - f\n    bukan_nol = a != 0\n    hasil = {\n        "n": int(len(a)),\n        "MAE": float(np.mean(np.abs(e))),\n        "RMSE": float(np.sqrt(np.mean(e ** 2))),\n        "MAPE (%)": float(np.mean(np.abs(e[bukan_nol] / a[bukan_nol])) * 100) if bukan_nol.any() else np.nan,\n        "Aktual = 0 (dikecualikan dari MAPE)": int((~bukan_nol).sum()),\n    }\n    penyebut = (np.abs(a) + np.abs(f)) / 2\n    ok = penyebut != 0\n    hasil["sMAPE (%)"] = float(np.mean(np.abs(e[ok]) / penyebut[ok]) * 100) if ok.any() else np.nan\n    if len(a) > 1:\n        dasar = a[:-1]\n        ok = dasar != 0\n        pemb = np.sum(((f[1:][ok] - a[1:][ok]) / dasar[ok]) ** 2)\n        peny = np.sum(((a[1:][ok] - dasar[ok]) / dasar[ok]) ** 2)\n        hasil["Theil U2"] = float(np.sqrt(pemb / peny)) if peny > 0 else np.nan\n    return hasil\n\n\ndef tabel_akurasi(aktual: pd.Series, ramalan: dict[str, pd.Series]) -> pd.DataFrame:\n    urutan = ["MAE", "RMSE", "MAPE (%)", "sMAPE (%)", "Theil U2", "n", "Aktual = 0 (dikecualikan dari MAPE)"]\n    tabel = pd.DataFrame({nama: ukuran_akurasi(aktual, f.reindex(aktual.index)) for nama, f in ramalan.items()}).T\n    return tabel[[c for c in urutan if c in tabel.columns]]\n',
    'modules/pipeline.py': '"""Satu pintu untuk menjalankan seluruh alur: validasi → backtest → proyeksi → evaluasi."""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\n\nimport pandas as pd\n\nfrom . import arima_model as am\nfrom . import evaluation as ev\nfrom . import forecasting as fc\nfrom .data_preprocessing import cek_transformasi, ke_skala_asli, ke_skala_model, label_periode\n\n\n@dataclass\nclass Pengaturan:\n    spek: am.SpesifikasiModel\n    transformasi: str                 # "log" | "sudah_log" | "tanpa"\n    awal_latih: pd.Period\n    akhir_latih: pd.Period\n    akhir_uji: pd.Period | None       # None = tanpa periode uji\n    horizon: int = 12\n    tingkat_kepercayaan: int = 95\n    koreksi_bias: bool = False\n\n\n@dataclass\nclass HasilProyeksi:\n    pengaturan: Pengaturan\n    y_asli: pd.Series\n    backtest: dict | None\n    akurasi: pd.DataFrame | None\n    hasil_akhir: object\n    koefisien_akhir: pd.DataFrame\n    statistik_akhir: pd.Series\n    diagnostik_akhir: dict\n    ramalan: pd.DataFrame\n    akhir_data_proyeksi: pd.Period\n    peringatan: list[str] = field(default_factory=list)\n\n\nclass GalatPengaturan(Exception):\n    pass\n\n\ndef periksa(seri: pd.Series, peng: Pengaturan) -> list[str]:\n    galat = list(cek_transformasi(seri.loc[peng.awal_latih:], peng.transformasi))\n    if peng.akhir_latih < peng.awal_latih:\n        galat.append("Akhir data latih harus setelah awal data latih.")\n    if peng.akhir_uji is not None and peng.akhir_uji <= peng.akhir_latih:\n        galat.append("Akhir data uji harus setelah akhir data latih.")\n    n_latih = int(seri.loc[peng.awal_latih:peng.akhir_latih].notna().sum())\n    minimal = peng.spek.minimal_observasi()\n    if n_latih < minimal:\n        galat.append(f"Data latih hanya {n_latih} observasi; model {peng.spek.label()} butuh minimal ±{minimal}. "\n                     "Perpanjang data latih atau sederhanakan model.")\n    if peng.horizon < 1:\n        galat.append("Horizon proyeksi minimal 1 periode.")\n    return galat\n\n\ndef jalankan(seri: pd.Series, peng: Pengaturan) -> HasilProyeksi:\n    galat = periksa(seri, peng)\n    if galat:\n        raise GalatPengaturan("\\n".join(galat))\n\n    alpha = 1 - peng.tingkat_kepercayaan / 100\n    y_model = ke_skala_model(seri, peng.transformasi)\n    y_asli = ke_skala_asli(y_model, peng.transformasi)\n    peringatan = []\n\n    bt, akurasi = None, None\n    if peng.akhir_uji is not None:\n        bt = fc.backtest(y_model, peng.spek, peng.awal_latih, peng.akhir_latih, peng.akhir_uji,\n                         peng.transformasi, alpha, peng.koreksi_bias)\n        aktual = bt["tabel"]["aktual"]\n        akurasi = ev.tabel_akurasi(aktual, {peng.spek.label(): bt["tabel"]["ramalan"], **bt["pembanding"]})\n        if aktual.isna().any():\n            peringatan.append(f"{int(aktual.isna().sum())} periode uji tanpa nilai aktual tidak dihitung di metrik.")\n\n    # Proyeksi ke depan: estimasi ulang dengan data s.d. observasi terakhir yang terisi\n    akhir_data = y_model.loc[peng.awal_latih:].last_valid_index()\n    y_final = y_model.loc[peng.awal_latih:akhir_data]\n    hasil_akhir = am.estimasi(y_final, peng.spek)\n    stat = am.statistik_ringkas(hasil_akhir, y_final)\n    ramalan = fc.proyeksi(hasil_akhir, peng.horizon, peng.spek, peng.transformasi, alpha, peng.koreksi_bias)\n    if peng.akhir_uji is not None and akhir_data > peng.akhir_latih:\n        peringatan.append(f"Proyeksi ke depan memakai model yang diestimasi ulang dengan data "\n                          f"{label_periode(peng.awal_latih)} s.d. {label_periode(akhir_data)} "\n                          "(termasuk periode uji), sehingga koefisiennya bisa berbeda dari model backtest.")\n\n    return HasilProyeksi(\n        pengaturan=peng, y_asli=y_asli, backtest=bt, akurasi=akurasi,\n        hasil_akhir=hasil_akhir, koefisien_akhir=am.tabel_koefisien(hasil_akhir),\n        statistik_akhir=stat, diagnostik_akhir=am.diagnostik(hasil_akhir, peng.spek),\n        ramalan=ramalan, akhir_data_proyeksi=akhir_data, peringatan=peringatan,\n    )\n\n\ndef tabel_hasil(h: HasilProyeksi) -> pd.DataFrame:\n    """Tabel gabungan: periode uji (aktual, ramalan, galat) + proyeksi ke depan."""\n    bagian = []\n    if h.backtest is not None:\n        t = h.backtest["tabel"].copy()\n        t.insert(0, "Jenis", "Uji (backtest)")\n        bagian.append(t)\n    r = h.ramalan[["ramalan", "batas_bawah", "batas_atas"]].copy()\n    r.insert(0, "Jenis", "Proyeksi ke depan")\n    r["aktual"], r["galat"] = float("nan"), float("nan")\n    bagian.append(r)\n    tabel = pd.concat(bagian)[["Jenis", "aktual", "ramalan", "galat", "batas_bawah", "batas_atas"]]\n    tabel.index = [label_periode(p) for p in tabel.index]\n    tabel.index.name = "Periode"\n    return tabel.rename(columns={"aktual": "Aktual", "ramalan": "Ramalan", "galat": "Galat (aktual − ramalan)",\n                                 "batas_bawah": "Batas bawah", "batas_atas": "Batas atas"})\n',
    'modules/visualization.py': '"""Grafik: Plotly untuk layar (interaktif), Matplotlib untuk file PNG unduhan.\n\nPlotly diimpor di dalam fungsi agar fungsi PNG bisa diuji tanpa Plotly."""\nfrom __future__ import annotations\n\nimport io\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt  # noqa: E402\nimport pandas as pd  # noqa: E402\n\nWARNA = {\n    "latih": "#1f2937",\n    "uji": "#0f766e",\n    "backtest": "#d97706",\n    "proyeksi": "#2563eb",\n    "pita_backtest": "rgba(217,119,6,0.15)",\n    "pita_proyeksi": "rgba(37,99,235,0.15)",\n}\nTATA_LETAK = dict(template="plotly_white", height=440, margin=dict(t=40, b=10, l=10, r=10),\n                  legend=dict(orientation="h", y=-0.15), hovermode="x unified")\n\n\ndef _x(index: pd.PeriodIndex):\n    return index.to_timestamp()\n\n\ndef grafik_seri(seri: pd.Series, judul: str = "", label_y: str = "Nilai"):\n    import plotly.graph_objects as go\n    fig = go.Figure(go.Scatter(x=_x(seri.index), y=seri, mode="lines+markers", name=seri.name or "Data",\n                               line=dict(color=WARNA["latih"]), marker=dict(size=4)))\n    fig.update_layout(title=judul, yaxis_title=label_y, **TATA_LETAK)\n    return fig\n\n\ndef _pita(fig, index, bawah, atas, warna, nama):\n    fig.add_scatter(x=_x(index), y=atas, line=dict(width=0), showlegend=False, hoverinfo="skip")\n    fig.add_scatter(x=_x(index), y=bawah, fill="tonexty", fillcolor=warna, line=dict(width=0), name=nama,\n                    hoverinfo="skip")\n\n\ndef grafik_hasil(h, label_y: str = "Nilai"):\n    import plotly.graph_objects as go\n    """Data latih, data uji, ramalan backtest, proyeksi ke depan, dan selang kepercayaan."""\n    peng = h.pengaturan\n    tingkat = peng.tingkat_kepercayaan\n    fig = go.Figure()\n    latih = h.y_asli.loc[peng.awal_latih:peng.akhir_latih]\n    fig.add_scatter(x=_x(latih.index), y=latih, name="Aktual — data latih", mode="lines",\n                    line=dict(color=WARNA["latih"]))\n    if h.backtest is not None:\n        t = h.backtest["tabel"]\n        _pita(fig, t.index, t["batas_bawah"], t["batas_atas"], WARNA["pita_backtest"], f"Selang {tingkat}% (uji)")\n        fig.add_scatter(x=_x(t.index), y=t["aktual"], name="Aktual — data uji", mode="lines+markers",\n                        line=dict(color=WARNA["uji"]))\n        fig.add_scatter(x=_x(t.index), y=t["ramalan"], name="Ramalan pada periode uji", mode="lines+markers",\n                        line=dict(color=WARNA["backtest"], dash="dash"))\n    sesudah = h.y_asli.loc[(peng.akhir_uji or peng.akhir_latih) + 1:h.akhir_data_proyeksi]\n    if len(sesudah.dropna()):\n        fig.add_scatter(x=_x(sesudah.index), y=sesudah, name="Aktual — setelah periode uji", mode="lines",\n                        line=dict(color=WARNA["latih"], dash="dot"))\n    r = h.ramalan\n    _pita(fig, r.index, r["batas_bawah"], r["batas_atas"], WARNA["pita_proyeksi"], f"Selang {tingkat}% (proyeksi)")\n    fig.add_scatter(x=_x(r.index), y=r["ramalan"], name="Proyeksi ke depan", mode="lines+markers",\n                    line=dict(color=WARNA["proyeksi"]))\n    fig.update_layout(title=f"{peng.spek.label()} — aktual dan ramalan", yaxis_title=label_y, **TATA_LETAK)\n    return fig\n\n\ndef grafik_acf(diag: dict):\n    import plotly.graph_objects as go\n    a = diag["acf"]\n    fig = go.Figure(go.Bar(x=a["lag"], y=a["acf"], marker_color=WARNA["proyeksi"], name="ACF"))\n    for tanda in (1, -1):\n        fig.add_hline(y=tanda * diag["batas_acf"], line_dash="dash", line_color="gray")\n    fig.update_layout(title="ACF residual (garis putus = batas signifikansi 95%)", xaxis_title="Lag",\n                      **{**TATA_LETAK, "height": 320, "hovermode": "closest"})\n    return fig\n\n\ndef png_hasil(h, label_y: str = "Nilai") -> bytes:\n    """Versi Matplotlib dari grafik_hasil, untuk diunduh sebagai PNG."""\n    peng = h.pengaturan\n    fig, ax = plt.subplots(figsize=(11, 5), dpi=150)\n    latih = h.y_asli.loc[peng.awal_latih:peng.akhir_latih]\n    ax.plot(_x(latih.index), latih, color=WARNA["latih"], label="Aktual — data latih")\n    if h.backtest is not None:\n        t = h.backtest["tabel"]\n        ax.fill_between(_x(t.index), t["batas_bawah"], t["batas_atas"], color=WARNA["backtest"], alpha=0.15)\n        ax.plot(_x(t.index), t["aktual"], color=WARNA["uji"], marker="o", ms=3, label="Aktual — data uji")\n        ax.plot(_x(t.index), t["ramalan"], color=WARNA["backtest"], ls="--", marker="o", ms=3,\n                label="Ramalan pada periode uji")\n    sesudah = h.y_asli.loc[(peng.akhir_uji or peng.akhir_latih) + 1:h.akhir_data_proyeksi]\n    if len(sesudah.dropna()):\n        ax.plot(_x(sesudah.index), sesudah, color=WARNA["latih"], ls=":", label="Aktual — setelah periode uji")\n    r = h.ramalan\n    ax.fill_between(_x(r.index), r["batas_bawah"], r["batas_atas"], color=WARNA["proyeksi"], alpha=0.15,\n                    label=f"Selang {peng.tingkat_kepercayaan}%")\n    ax.plot(_x(r.index), r["ramalan"], color=WARNA["proyeksi"], marker="o", ms=3, label="Proyeksi ke depan")\n    ax.set_title(f"{peng.spek.label()} — aktual dan ramalan")\n    ax.set_ylabel(label_y)\n    ax.grid(alpha=0.3)\n    ax.legend(loc="upper left", fontsize=8)\n    ax.ticklabel_format(axis="y", style="plain")\n    fig.autofmt_xdate()\n    buf = io.BytesIO()\n    fig.savefig(buf, format="png", bbox_inches="tight")\n    plt.close(fig)\n    return buf.getvalue()\n',
    'modules/export_results.py': '"""Membuat file unduhan dari hasil model yang benar-benar dijalankan."""\nfrom __future__ import annotations\n\nimport io\nfrom datetime import datetime\n\nimport pandas as pd\n\nfrom .data_preprocessing import TRANSFORMASI, label_periode\nfrom .pipeline import HasilProyeksi, tabel_hasil\n\n\ndef tabel_pengaturan(h: HasilProyeksi) -> pd.DataFrame:\n    p = h.pengaturan\n    baris = {\n        "Model": p.spek.label(),\n        "Order (p,d,q)": str(p.spek.order),\n        "Seasonal order (P,D,Q,s)": str(p.spek.seasonal_order),\n        "Konstanta C": "Ya" if p.spek.konstanta else "Tidak",\n        "Transformasi": TRANSFORMASI[p.transformasi],\n        "Data latih": f"{label_periode(p.awal_latih)} s.d. {label_periode(p.akhir_latih)}",\n        "Data uji": f"{label_periode(p.akhir_latih + 1)} s.d. {label_periode(p.akhir_uji)}" if p.akhir_uji else "-",\n        "Data untuk proyeksi": f"{label_periode(p.awal_latih)} s.d. {label_periode(h.akhir_data_proyeksi)}",\n        "Horizon": p.horizon,\n        "Tingkat kepercayaan": f"{p.tingkat_kepercayaan}%",\n        "Koreksi bias exp()": "Ya" if p.koreksi_bias else "Tidak",\n        "Paksa stasioner / invertible": f"{p.spek.paksa_stasioner} / {p.spek.paksa_invertible}",\n        "Dibuat": datetime.now().strftime("%Y-%m-%d %H:%M"),\n    }\n    return pd.DataFrame({"Nilai": baris}).rename_axis("Pengaturan")\n\n\ndef csv_hasil(h: HasilProyeksi) -> bytes:\n    return tabel_hasil(h).to_csv().encode("utf-8-sig")  # utf-8-sig agar terbaca benar di Excel\n\n\ndef excel_laporan(h: HasilProyeksi) -> bytes:\n    buf = io.BytesIO()\n    with pd.ExcelWriter(buf, engine="openpyxl") as w:\n        tabel_pengaturan(h).to_excel(w, sheet_name="Pengaturan")\n        tabel_hasil(h).to_excel(w, sheet_name="Hasil")\n        if h.akurasi is not None:\n            h.akurasi.to_excel(w, sheet_name="Evaluasi")\n        h.koefisien_akhir.to_excel(w, sheet_name="Koefisien")\n        h.statistik_akhir.to_frame("Nilai").to_excel(w, sheet_name="Statistik model")\n        if h.backtest is not None:\n            from .arima_model import tabel_koefisien\n            tabel_koefisien(h.backtest["hasil"]).to_excel(w, sheet_name="Koefisien backtest")\n    return buf.getvalue()\n\n\ndef excel_hasil_saja(h: HasilProyeksi) -> bytes:\n    buf = io.BytesIO()\n    with pd.ExcelWriter(buf, engine="openpyxl") as w:\n        tabel_hasil(h).to_excel(w, sheet_name="Hasil")\n    return buf.getvalue()\n\n\ndef teks_ringkasan(h: HasilProyeksi) -> bytes:\n    from .arima_model import ringkasan_teks\n    bagian = ["ARIMA Forecasting Platform — ringkasan model", "=" * 60, "",\n              tabel_pengaturan(h).to_string(), "",\n              "MODEL UNTUK PROYEKSI KE DEPAN", "-" * 60,\n              ringkasan_teks(h.hasil_akhir, h.pengaturan.spek, h.statistik_akhir)]\n    if h.diagnostik_akhir["peringatan"]:\n        bagian += ["", "PERINGATAN MODEL"] + [f"- {p}" for p in h.diagnostik_akhir["peringatan"]]\n    if h.akurasi is not None:\n        bagian += ["", "EVALUASI PERIODE UJI", "-" * 60, h.akurasi.to_string(float_format=lambda v: f"{v:,.4f}")]\n    if h.peringatan:\n        bagian += ["", "CATATAN"] + [f"- {p}" for p in h.peringatan]\n    return "\\n".join(bagian).encode("utf-8")\n',
    'reference/arima_engine.py': '"""\nMesin proyeksi SARIMA untuk outflow (dimodelkan dalam log), disusun agar\nsedekat mungkin dengan pengaturan EViews:\n\n- Y = log(outflow)\n- Konstanta "C" diperlakukan seperti EViews: sebagai RATA-RATA (regresi pada\n  konstanta dengan galat ARMA), bukan intercept persamaan AR.\n- Estimasi Maximum Likelihood, kovarians koefisien dengan outer product of\n  gradients (OPG) seperti keluaran EViews.\n- Kriteria informasi (AIC/SC/HQ) dilaporkan per observasi, seperti EViews.\n\nModul ini tidak bergantung pada Streamlit, jadi bisa dipakai dari skrip biasa.\n"""\nfrom __future__ import annotations\n\nimport re\nfrom dataclasses import dataclass, field\n\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats\n\n# ---------------------------------------------------------------------------\n# 1. Membaca dan memvalidasi data\n# ---------------------------------------------------------------------------\n\n_POLA_EVIEWS = re.compile(r"^\\s*(\\d{4})\\s*[Mm]\\s*(\\d{1,2})\\s*$")  # contoh: 2021M01\n\n\ndef baca_file(file) -> pd.DataFrame:\n    """Baca CSV atau Excel. `file` boleh path (str) atau objek unggahan Streamlit."""\n    nama = getattr(file, "name", str(file)).lower()\n    if nama.endswith((".xlsx", ".xls")):\n        return pd.read_excel(file)\n    # CSV: coba deteksi pemisah (koma atau titik koma)\n    return pd.read_csv(file, sep=None, engine="python")\n\n\ndef ke_periode_bulanan(kolom: pd.Series) -> pd.PeriodIndex:\n    """Ubah kolom tanggal (2021M01, 2021-01, 01/01/2021, tanggal Excel) ke periode bulanan."""\n    teks = kolom.astype(str).str.strip()\n    if teks.str.match(_POLA_EVIEWS).all():\n        bagian = teks.str.extract(_POLA_EVIEWS)\n        label = bagian[0] + "-" + bagian[1].str.zfill(2)\n        return pd.PeriodIndex(label, freq="M")\n    tanggal = pd.to_datetime(kolom, errors="raise")\n    return pd.PeriodIndex(tanggal, freq="M")\n\n\ndef _ke_angka(kolom: pd.Series) -> pd.Series:\n    """Ubah teks angka ke float. Menangani format Indonesia (1.234.567,89)."""\n    if pd.api.types.is_numeric_dtype(kolom):\n        return kolom.astype(float)\n    teks = kolom.astype(str).str.strip()\n    format_id = teks.str.contains(r",\\d+$") | teks.str.contains(r"\\.\\d{3}(?:\\.|$)")\n    teks = teks.where(~format_id, teks.str.replace(".", "", regex=False).str.replace(",", ".", regex=False))\n    return pd.to_numeric(teks, errors="coerce")\n\n\n@dataclass\nclass HasilValidasi:\n    seri: pd.Series                       # seri bulanan (level, bukan log)\n    galat: list[str] = field(default_factory=list)       # menghentikan proses\n    peringatan: list[str] = field(default_factory=list)  # boleh lanjut\n\n    @property\n    def ok(self) -> bool:\n        return not self.galat\n\n\ndef siapkan_seri(df: pd.DataFrame, kolom_tanggal: str, kolom_nilai: str) -> HasilValidasi:\n    """Bentuk seri bulanan dan periksa masalah umum pada data."""\n    galat, peringatan = [], []\n    try:\n        periode = ke_periode_bulanan(df[kolom_tanggal])\n    except Exception as e:  # format tanggal tidak dikenali\n        return HasilValidasi(pd.Series(dtype=float), [f"Kolom tanggal tidak terbaca: {e}"])\n\n    nilai = _ke_angka(df[kolom_nilai])\n    seri = pd.Series(nilai.to_numpy(), index=periode, name=kolom_nilai).sort_index()\n\n    if seri.index.duplicated().any():\n        dup = sorted({str(p) for p in seri.index[seri.index.duplicated()]})\n        galat.append(f"Ada bulan ganda: {\', \'.join(dup)}")\n        seri = seri[~seri.index.duplicated(keep="first")]\n\n    lengkap = pd.period_range(seri.index.min(), seri.index.max(), freq="M")\n    hilang = lengkap.difference(seri.index)\n    if len(hilang):\n        peringatan.append(\n            f"Ada {len(hilang)} bulan yang tidak ada di data: "\n            + ", ".join(str(p) for p in hilang[:12]) + (" ..." if len(hilang) > 12 else "")\n            + ". Bulan ini dianggap kosong (missing)."\n        )\n    seri = seri.reindex(lengkap)\n\n    kosong = seri.isna() & ~seri.index.isin(hilang)\n    if kosong.any():\n        peringatan.append(f"Ada {int(kosong.sum())} nilai kosong/tidak terbaca sebagai angka.")\n\n    tidak_positif = seri <= 0\n    if tidak_positif.any():\n        bulan = ", ".join(str(p) for p in seri.index[tidak_positif][:12])\n        galat.append(f"Nilai harus > 0 karena akan di-log. Bermasalah di: {bulan}")\n\n    if seri.notna().sum() < 36:\n        peringatan.append("Data kurang dari 36 bulan; pola musiman 12 bulan sulit diestimasi dengan andal.")\n\n    return HasilValidasi(seri, galat, peringatan)\n\n\n# ---------------------------------------------------------------------------\n# 2. Spesifikasi dan estimasi model\n# ---------------------------------------------------------------------------\n\n@dataclass\nclass SpesifikasiModel:\n    order: tuple[int, int, int] = (0, 0, 0)\n    seasonal_order: tuple[int, int, int, int] = (1, 0, 1, 12)\n    konstanta: bool = True\n    # EViews tidak memaksa stasioner/invertible, jadi default False agar\n    # hasilnya bisa dibandingkan. Ubah ke True untuk estimasi yang "aman".\n    paksa_stasioner: bool = False\n    paksa_invertible: bool = False\n\n    def label(self) -> str:\n        p, d, q = self.order\n        P, D, Q, s = self.seasonal_order\n        return f"SARIMA({p},{d},{q})({P},{D},{Q}){s}" + (" + C" if self.konstanta else "")\n\n\ndef _exog_konstanta(index, pakai: bool):\n    if not pakai:\n        return None\n    return pd.DataFrame({"C": np.ones(len(index))}, index=index)\n\n\ndef estimasi(log_y: pd.Series, spek: SpesifikasiModel):\n    """Estimasi SARIMA dengan ML. Mengembalikan objek hasil statsmodels."""\n    from statsmodels.tsa.statespace.sarimax import SARIMAX  # impor di sini agar modul tetap bisa dimuat tanpa statsmodels\n\n    model = SARIMAX(\n        log_y,\n        exog=_exog_konstanta(log_y.index, spek.konstanta),\n        order=spek.order,\n        seasonal_order=spek.seasonal_order,\n        trend="n",\n        enforce_stationarity=spek.paksa_stasioner,\n        enforce_invertibility=spek.paksa_invertible,\n    )\n    return model.fit(disp=False, maxiter=2000, cov_type="opg")\n\n\n_NAMA_EVIEWS = {"sigma2": "SIGMASQ"}\n\n\ndef _nama_parameter(nama: str) -> str:\n    if nama in _NAMA_EVIEWS:\n        return _NAMA_EVIEWS[nama]\n    m = re.match(r"^(ar|ma)\\.(S\\.)?L(\\d+)$", nama)\n    if m:\n        jenis, musiman, lag = m.groups()\n        return f"{\'S\' if musiman else \'\'}{jenis.upper()}({lag})"\n    return nama\n\n\ndef tabel_koefisien(hasil) -> pd.DataFrame:\n    tabel = pd.DataFrame({\n        "Koefisien": hasil.params,\n        "Std. Error": hasil.bse,\n        "z-Statistic": hasil.tvalues,\n        "Prob.": hasil.pvalues,\n    })\n    tabel.index = [_nama_parameter(n) for n in tabel.index]\n    tabel.index.name = "Variabel"\n    return tabel\n\n\ndef _residual_efektif(hasil) -> pd.Series:\n    """Residual tanpa periode awal (burn-in) inisialisasi difus, jika ada."""\n    burn = int(getattr(hasil, "loglikelihood_burn", 0) or 0)\n    return pd.Series(hasil.resid).iloc[burn:].dropna()\n\n\ndef statistik_ringkas(hasil, log_y: pd.Series) -> pd.Series:\n    """Statistik ringkas bergaya EViews (kriteria informasi per observasi)."""\n    n = int(hasil.nobs)\n    k = len(hasil.params)  # EViews ikut menghitung SIGMASQ\n    ll = float(hasil.llf)\n    resid = _residual_efektif(hasil)\n    y_sejajar = log_y.loc[resid.index] if isinstance(resid.index, pd.PeriodIndex) else log_y.iloc[-len(resid):]\n    ssr = float((resid ** 2).sum())\n    sst = float(((y_sejajar - y_sejajar.mean()) ** 2).sum())\n    return pd.Series({\n        "Jumlah observasi": n,\n        "Log likelihood": ll,\n        "Akaike info criterion": (-2 * ll + 2 * k) / n,\n        "Schwarz criterion": (-2 * ll + k * np.log(n)) / n,\n        "Hannan-Quinn criter.": (-2 * ll + 2 * k * np.log(np.log(n))) / n,\n        "R-squared (perkiraan)": 1 - ssr / sst if sst > 0 else np.nan,\n        "Sum squared resid": ssr,\n        "Observasi burn-in (dikecualikan dari residual)": int(getattr(hasil, "loglikelihood_burn", 0) or 0),\n    })\n\n\n# ---------------------------------------------------------------------------\n# 3. Diagnostik\n# ---------------------------------------------------------------------------\n\ndef diagnostik(hasil, spek: SpesifikasiModel) -> dict:\n    from statsmodels.stats.diagnostic import acorr_ljungbox\n    from statsmodels.tsa.stattools import acf\n\n    resid = _residual_efektif(hasil)\n    p, _, q = spek.order\n    P, _, Q, _ = spek.seasonal_order\n    df_model = p + q + P + Q\n    lag_uji = [lag for lag in (12, 24) if lag < len(resid)]\n    ljung_box = acorr_ljungbox(resid, lags=lag_uji, model_df=df_model) if lag_uji else pd.DataFrame()\n\n    jb = stats.jarque_bera(resid)\n    nlags = min(24, len(resid) - 1)\n    nilai_acf = acf(resid, nlags=nlags, fft=False)[1:]\n    batas_acf = 1.96 / np.sqrt(len(resid))\n\n    akar_ar = 1 / np.abs(hasil.arroots) if len(hasil.arroots) else np.array([])\n    akar_ma = 1 / np.abs(hasil.maroots) if len(hasil.maroots) else np.array([])\n\n    return {\n        "ljung_box": ljung_box,\n        "jarque_bera": {"statistik": float(jb.statistic), "p_value": float(jb.pvalue)},\n        "acf": pd.DataFrame({"lag": np.arange(1, nlags + 1), "acf": nilai_acf}),\n        "batas_acf": batas_acf,\n        "modulus_akar_ar_invers": akar_ar,\n        "modulus_akar_ma_invers": akar_ma,\n        "peringatan": peringatan_model(hasil, akar_ar, akar_ma),\n    }\n\n\ndef peringatan_model(hasil, akar_ar, akar_ma, ambang: float = 0.99) -> list[str]:\n    """Deteksi masalah yang membuat model tidak bisa ditafsirkan seperti ARIMA biasa."""\n    pesan = []\n    if len(akar_ar) and akar_ar.max() >= ambang:\n        pesan.append(\n            f"Akar AR invers = {akar_ar.max():.4f} (≥ {ambang}). Komponen AR berada di/dekat unit root, "\n            "sehingga model tidak stasioner. Nilai C (rata-rata) praktis tidak teridentifikasi."\n        )\n    if len(akar_ma) and akar_ma.max() >= ambang:\n        pesan.append(\n            f"Akar MA invers = {akar_ma.max():.4f} (≥ {ambang}). Komponen MA tidak/nyaris tidak invertible."\n        )\n    params = hasil.params\n    for ar, ma in (("ar.S.L12", "ma.S.L12"), ("ar.L1", "ma.L1")):\n        if ar in params.index and ma in params.index:\n            phi, theta = params[ar], params[ma]\n            if abs(phi + theta) < 0.05 and abs(phi) > 0.5:\n                pesan.append(\n                    f"{_nama_parameter(ar)} = {phi:.4f} dan {_nama_parameter(ma)} = {theta:.4f} hampir saling "\n                    "meniadakan (pembatalan akar). Model ini pada dasarnya setara dengan pola musiman "\n                    "deterministik (rata-rata per bulan) + white noise; t-statistik yang sangat besar "\n                    "tidak bermakna."\n                )\n    se = hasil.bse\n    if (se.drop("sigma2", errors="ignore") < 1e-3).any():\n        pesan.append("Ada standard error yang sangat kecil (< 0,001): tanda estimasi berada di batas ruang parameter.")\n    return pesan\n\n\n# ---------------------------------------------------------------------------\n# 4. Ukuran akurasi dan baseline\n# ---------------------------------------------------------------------------\n\ndef ukuran_akurasi(aktual, ramalan) -> dict:\n    a = np.asarray(aktual, dtype=float)\n    f = np.asarray(ramalan, dtype=float)\n    galat = a - f\n    hasil = {\n        "MAPE (%)": float(np.mean(np.abs(galat / a)) * 100),\n        "sMAPE (%)": float(np.mean(np.abs(galat) / ((np.abs(a) + np.abs(f)) / 2)) * 100),\n        "MAE": float(np.mean(np.abs(galat))),\n        "RMSE": float(np.sqrt(np.mean(galat ** 2))),\n    }\n    if len(a) > 1:\n        pembilang = np.sum(((f[1:] - a[1:]) / a[:-1]) ** 2)\n        penyebut = np.sum(((a[1:] - a[:-1]) / a[:-1]) ** 2)\n        hasil["Theil U2"] = float(np.sqrt(pembilang / penyebut)) if penyebut > 0 else np.nan\n    return hasil\n\n\ndef baseline_naif_musiman(y_latih: pd.Series, periode_uji: pd.PeriodIndex) -> pd.Series:\n    """Ramalan = nilai bulan yang sama setahun sebelumnya (hanya dari data latih)."""\n    gabung = y_latih.copy()\n    for p in periode_uji:\n        gabung.loc[p] = gabung.get(p - 12, np.nan)\n    return gabung.loc[periode_uji].rename("Naif musiman")\n\n\ndef baseline_rata_bulanan(y_latih: pd.Series, periode_uji: pd.PeriodIndex) -> pd.Series:\n    """Ramalan = rata-rata geometrik (rata-rata log) per bulan kalender di data latih."""\n    rata = np.log(y_latih.dropna()).groupby(y_latih.dropna().index.month).mean()\n    return pd.Series(np.exp(rata.reindex(periode_uji.month).to_numpy()), index=periode_uji,\n                     name="Rata-rata per bulan")\n\n\n# ---------------------------------------------------------------------------\n# 5. Proyeksi dan backtest\n# ---------------------------------------------------------------------------\n\ndef proyeksi(hasil, langkah: int, spek: SpesifikasiModel, alpha: float = 0.05,\n             koreksi_bias: bool = False) -> pd.DataFrame:\n    """Proyeksi dinamis `langkah` bulan ke depan, dikembalikan ke skala Rupiah (exp)."""\n    akhir = pd.Period(hasil.fittedvalues.index[-1], "M")\n    index_depan = pd.period_range(akhir + 1, periods=langkah, freq="M")\n    ramalan = hasil.get_forecast(steps=langkah, exog=_exog_konstanta(index_depan, spek.konstanta))\n    rata_log = np.asarray(ramalan.predicted_mean)\n    se_log = np.asarray(ramalan.se_mean)\n    ci = np.asarray(ramalan.conf_int(alpha=alpha))\n    faktor = np.exp(se_log ** 2 / 2) if koreksi_bias else 1.0\n    return pd.DataFrame({\n        "log_proyeksi": rata_log,\n        "se_log": se_log,\n        "proyeksi": np.exp(rata_log) * faktor,\n        "batas_bawah": np.exp(ci[:, 0]),\n        "batas_atas": np.exp(ci[:, 1]),\n    }, index=index_depan)\n\n\ndef potong(seri: pd.Series, awal, akhir) -> pd.Series:\n    return seri.loc[pd.Period(awal, "M"):pd.Period(akhir, "M")]\n\n\ndef backtest(y: pd.Series, spek: SpesifikasiModel, awal_latih, akhir_latih, akhir_uji,\n             alpha: float = 0.05, koreksi_bias: bool = False) -> dict:\n    """Estimasi di data latih, ramal dinamis sepanjang periode uji, bandingkan dengan aktual."""\n    y_latih = potong(y, awal_latih, akhir_latih)\n    y_uji = potong(y, pd.Period(akhir_latih, "M") + 1, akhir_uji)\n    hasil = estimasi(np.log(y_latih), spek)\n    ramal = proyeksi(hasil, len(y_uji), spek, alpha, koreksi_bias)\n\n    tabel = pd.DataFrame({\n        "aktual": y_uji,\n        spek.label(): ramal["proyeksi"],\n        "batas_bawah": ramal["batas_bawah"],\n        "batas_atas": ramal["batas_atas"],\n        "Naif musiman": baseline_naif_musiman(y_latih, y_uji.index),\n        "Rata-rata per bulan": baseline_rata_bulanan(y_latih, y_uji.index),\n    })\n    sah = tabel["aktual"].notna()\n    akurasi = pd.DataFrame({\n        nama: ukuran_akurasi(tabel.loc[sah, "aktual"], tabel.loc[sah, nama])\n        for nama in (spek.label(), "Naif musiman", "Rata-rata per bulan")\n    }).T\n    return {"hasil": hasil, "tabel": tabel, "akurasi": akurasi}\n'
}

for nama, isi in BERKAS_MESIN.items():
    path = FOLDER / nama
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(isi)

# Data contoh SINTETIS (bukan data asli) — untuk mencoba notebook
import numpy as np, pandas as pd
rng = np.random.default_rng(7)
_per = pd.period_range("2021-01", "2026-06", freq="M")
_pola = {1:-0.6, 2:-0.3, 3:0.4, 4:0.6, 5:0.3, 6:0.0, 7:-0.1, 8:0.0, 9:-0.2, 10:-0.1, 11:0.1, 12:0.7}
_logy = 12.6 + np.array([_pola[p.month] for p in _per]) + rng.normal(0, 0.45, len(_per))
_contoh = pd.DataFrame({"tanggal": _per.to_timestamp(how="end").normalize().date, "outflow": np.round(np.exp(_logy))})
(FOLDER / "sample_data").mkdir(exist_ok=True)
_contoh.to_excel(FOLDER / "sample_data" / "sample_dataset.xlsx", index=False)
_contoh.to_csv(FOLDER / "sample_data" / "sample_dataset.csv", index=False)

import modules.data_loader as dl, modules.data_preprocessing as dp, modules.exploration as ex
import modules.arima_model as am, modules.forecasting as fc, modules.evaluation as ev
import modules.pipeline as pl, modules.visualization as vz, modules.export_results as er
for _m in (dl, dp, ex, am, fc, ev, pl, vz, er):
    importlib.reload(_m)

import statsmodels, plotly
from IPython.display import display, Markdown
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
print(f"✅ Mesin siap. statsmodels {statsmodels.__version__}, pandas {pd.__version__}, plotly {plotly.__version__}")

In [ ]:
#@title Langkah 2 — Unggah data
#@markdown Centang untuk mencoba dengan data contoh sintetis. Hapus centang untuk mengunggah file Anda (.xlsx / .csv).
PAKAI_DATA_CONTOH = False #@param {type:"boolean"}

if PAKAI_DATA_CONTOH:
    NAMA_FILE = str(FOLDER / "sample_data" / "sample_dataset.xlsx")
    print("Memakai data contoh SINTETIS.")
else:
    from google.colab import files
    print("Klik 'Choose Files' lalu pilih file data Anda...")
    _up = files.upload()
    if not _up:
        raise SystemExit("Tidak ada file yang diunggah.")
    NAMA_FILE = str(FOLDER / list(_up)[0])

try:
    df_mentah = dl.baca_file(NAMA_FILE)
except dl.GalatData as e:
    raise SystemExit(f"❌ {e}")
ringkas = dl.ringkasan_dataset(df_mentah)
print(f"✅ {Path(NAMA_FILE).name}: {ringkas['jumlah_baris']} baris, {ringkas['jumlah_kolom']} kolom")
display(ringkas["info_kolom"])
display(df_mentah.head(10))

In [ ]:
#@title Langkah 3 — Pilih kolom & periksa data
#@markdown Tulis nama kolom persis seperti di file. **Kosongkan** agar dipilih otomatis.
KOLOM_TANGGAL = "" #@param {type:"string"}
KOLOM_TARGET = "" #@param {type:"string"}
FREKUENSI = "Bulanan" #@param ["Bulanan", "Triwulanan", "Mingguan", "Harian"]

_tgl, _tgt = dl.tebak_kolom(df_mentah)
KOLOM_TANGGAL = KOLOM_TANGGAL.strip() or _tgl
KOLOM_TARGET = KOLOM_TARGET.strip() or _tgt
for _k in (KOLOM_TANGGAL, KOLOM_TARGET):
    if _k not in df_mentah.columns:
        raise SystemExit(f"❌ Kolom '{_k}' tidak ada. Kolom yang tersedia: {list(df_mentah.columns)}")
print(f"Kolom tanggal: {KOLOM_TANGGAL} | Kolom target: {KOLOM_TARGET} | Frekuensi: {FREKUENSI}")

validasi = dp.siapkan_seri(df_mentah, KOLOM_TANGGAL, KOLOM_TARGET, FREKUENSI)
for _p in validasi.peringatan:
    print("⚠️", _p)
if not validasi.ok:
    raise SystemExit("❌ " + "\n❌ ".join(validasi.galat))
seri = validasi.seri
KODE_FREK, S_BAWAAN = dp.FREKUENSI[FREKUENSI]
print(f"✅ Data siap: {len(seri)} periode ({dp.label_periode(seri.index.min())} – {dp.label_periode(seri.index.max())}), "
      f"{int(seri.isna().sum())} kosong")

vz.grafik_seri(seri, f"{KOLOM_TARGET} ({FREKUENSI.lower()})", KOLOM_TARGET).show()
display(ex.statistik_deskriptif(seri).to_frame("Nilai"))

In [ ]:
#@title Langkah 4 — Pengaturan model
#@markdown **Transformasi**: `log` = data di-log dulu (data asli > 0); `sudah_log` = kolom sudah berupa log; `tanpa` = apa adanya.
TRANSFORMASI = "log" #@param ["log", "sudah_log", "tanpa"]
#@markdown **Periode** (format bulanan `2021-01`, triwulanan `2021Q1`, mingguan/harian `2021-01-04`). Kosongkan untuk otomatis; kosongkan AKHIR_UJI dan centang TANPA_UJI bila tidak ingin data uji.
AWAL_LATIH = "2021-01" #@param {type:"string"}
AKHIR_LATIH = "2025-06" #@param {type:"string"}
AKHIR_UJI = "2026-06" #@param {type:"string"}
TANPA_UJI = False #@param {type:"boolean"}
#@markdown **Orde model** SARIMA(p,d,q)(P,D,Q)s. Bawaan (0,0,0)(1,0,0)12 = SAR(12).
p = 0 #@param {type:"integer"}
d = 0 #@param {type:"integer"}
q = 0 #@param {type:"integer"}
P = 1 #@param {type:"integer"}
D = 0 #@param {type:"integer"}
Q = 0 #@param {type:"integer"}
s = 12 #@param {type:"integer"}
KONSTANTA = True #@param {type:"boolean"}
#@markdown **Proyeksi**
HORIZON = 12 #@param {type:"integer"}
TINGKAT_KEPERCAYAAN = 95 #@param [80, 90, 95, 99] {type:"raw"}
#@markdown **Opsi lanjutan** (biarkan mati agar sama dengan EViews)
PAKSA_STASIONER = False #@param {type:"boolean"}
PAKSA_INVERTIBLE = False #@param {type:"boolean"}
KOREKSI_BIAS = False #@param {type:"boolean"}

def _periode(teks, cadangan):
    teks = str(teks).strip()
    return pd.Period(teks, freq=KODE_FREK) if teks else cadangan

_terisi = seri.dropna().index
_n_uji = S_BAWAAN if len(_terisi) > 3 * S_BAWAAN else max(1, len(_terisi) // 5)
awal_latih = _periode(AWAL_LATIH, _terisi[0])
akhir_uji = None if TANPA_UJI else _periode(AKHIR_UJI, _terisi[-1])
akhir_latih = _periode(AKHIR_LATIH, _terisi[-1] if TANPA_UJI else _terisi[-1 - _n_uji])

for _g in dp.cek_transformasi(seri.loc[awal_latih:], TRANSFORMASI):
    raise SystemExit("❌ " + _g)
if (_h := dp.petunjuk_log_ganda(seri, TRANSFORMASI)):
    print("⚠️", _h)

spek = am.SpesifikasiModel(order=(p, d, q), seasonal_order=(P, D, Q, s), konstanta=KONSTANTA,
                           paksa_stasioner=PAKSA_STASIONER, paksa_invertible=PAKSA_INVERTIBLE)
pengaturan = pl.Pengaturan(spek=spek, transformasi=TRANSFORMASI, awal_latih=awal_latih, akhir_latih=akhir_latih,
                           akhir_uji=akhir_uji, horizon=HORIZON, tingkat_kepercayaan=int(TINGKAT_KEPERCAYAAN),
                           koreksi_bias=KOREKSI_BIAS)
_galat = pl.periksa(seri, pengaturan)
if _galat:
    raise SystemExit("❌ " + "\n❌ ".join(_galat))
print(f"✅ Model: {spek.label()} | transformasi: {TRANSFORMASI}")
print(f"   Latih: {dp.label_periode(awal_latih)} – {dp.label_periode(akhir_latih)}"
      + (f" | Uji: {dp.label_periode(akhir_latih + 1)} – {dp.label_periode(akhir_uji)}" if akhir_uji else " | tanpa uji")
      + f" | Horizon: {HORIZON}")

In [ ]:
#@title Langkah 5 — Jalankan SARIMA
try:
    hasil = pl.jalankan(seri, pengaturan)
except pl.GalatPengaturan as e:
    raise SystemExit("❌ " + str(e))

for _p in hasil.diagnostik_akhir["peringatan"]:
    print("⚠️", _p)
for _p in hasil.peringatan:
    print("ℹ️", _p)

if hasil.backtest is not None:
    display(Markdown(f"### Model backtest — data latih {dp.label_periode(awal_latih)} s.d. {dp.label_periode(akhir_latih)}\n"
                     "Dipakai untuk meramal periode uji. **Bandingkan model ini dengan EViews.**"))
    display(am.tabel_koefisien(hasil.backtest["hasil"]))
    display(am.statistik_ringkas(hasil.backtest["hasil"], hasil.backtest["y_latih"]).to_frame("Nilai"))

display(Markdown(f"### Model untuk proyeksi ke depan — data {dp.label_periode(awal_latih)} s.d. "
                 f"{dp.label_periode(hasil.akhir_data_proyeksi)}"))
display(hasil.koefisien_akhir)
display(hasil.statistik_akhir.to_frame("Nilai"))

In [ ]:
#@title Langkah 6 — Diagnostik residual (model proyeksi)
_dg = hasil.diagnostik_akhir
print("Ljung-Box (p > 0,05 = tidak ada autokorelasi residual tersisa):")
display(_dg["ljung_box"].rename(columns={"lb_stat": "Statistik Q", "lb_pvalue": "p-value"}))
_jb = _dg["jarque_bera"]
print(f"Jarque-Bera = {_jb['statistik']:.3f}, p = {_jb['p_value']:.4f} → residual "
      f"{'normal' if _jb['p_value'] > 0.05 else 'TIDAK normal'}")
if _dg["akar_ar_invers_maks"] is not None:
    print(f"Akar AR invers maksimum = {_dg['akar_ar_invers_maks']:.4f} (harus < 1)")
if _dg["akar_ma_invers_maks"] is not None:
    print(f"Akar MA invers maksimum = {_dg['akar_ma_invers_maks']:.4f} (harus < 1)")
vz.grafik_acf(_dg).show()

In [ ]:
#@title Langkah 7 — Grafik & tabel hasil
vz.grafik_hasil(hasil, KOLOM_TARGET).show()
tabel = pl.tabel_hasil(hasil)
display(tabel.style.format({c: "{:,.2f}" for c in tabel.columns if c != "Jenis"}, na_rep="—"))

In [ ]:
#@title Langkah 8 — Evaluasi periode uji
if hasil.akurasi is None:
    print("Tidak ada periode uji (TANPA_UJI dicentang), jadi akurasi tidak dapat diukur.")
else:
    display(Markdown("Akurasi **di luar sampel**: model hanya dilatih dengan data latih. "
                     "Pembanding: *Naif musiman* (nilai musim lalu) dan *Rata-rata musiman*."))
    display(hasil.akurasi)
    _nol = int(hasil.akurasi.iloc[0].get("Aktual = 0 (dikecualikan dari MAPE)", 0))
    if _nol:
        print(f"⚠️ {_nol} periode uji bernilai aktual 0 dan dikecualikan dari MAPE.")
    display(Markdown("\n".join(f"- **{k}**: {v}" for k, v in ev.PENJELASAN.items())))

In [ ]:
#@title Langkah 9 — Bandingkan dengan EViews (opsional)
#@markdown Isi dengan angka dari output EViews (model dan periode latih/uji yang SAMA). Bawaan: SAR(12) log outflow, latih 2021M01–2025M06.
EV_C = 12.682 #@param {type:"number"}
EV_SAR12 = 0.448 #@param {type:"number"}
EV_AIC = 2.450 #@param {type:"number"}
EV_SC = 2.560 #@param {type:"number"}
EV_MAPE = 76.0 #@param {type:"number"}
EV_SMAPE = 32.8 #@param {type:"number"}
EV_MAE = 87165 #@param {type:"number"}
EV_RMSE = 116741 #@param {type:"number"}
EV_THEIL_U2 = 0.36 #@param {type:"number"}

if hasil.backtest is None:
    raise SystemExit("Perbandingan butuh periode uji. Hapus centang TANPA_UJI di Langkah 4.")
_k = am.tabel_koefisien(hasil.backtest["hasil"])["Koefisien"]
_st = am.statistik_ringkas(hasil.backtest["hasil"], hasil.backtest["y_latih"])
_ak = hasil.akurasi.loc[spek.label()]
_baris = [  # nama, EViews, aplikasi, toleransi
    ("C", EV_C, _k.get("C"), 0.01),
    (f"SAR({s})", EV_SAR12, _k.get(f"SAR({s})"), 0.02),
    ("AIC (per obs.)", EV_AIC, _st["Akaike info criterion (per obs., EViews)"], 0.02),
    ("SC (per obs.)", EV_SC, _st["Schwarz criterion (per obs., EViews)"], 0.02),
    ("MAPE (%)", EV_MAPE, _ak.get("MAPE (%)"), 1.0),
    ("sMAPE (%)", EV_SMAPE, _ak.get("sMAPE (%)"), 1.0),
    ("MAE", EV_MAE, _ak.get("MAE"), 0.01 * EV_MAE),
    ("RMSE", EV_RMSE, _ak.get("RMSE"), 0.01 * EV_RMSE),
    ("Theil U2", EV_THEIL_U2, _ak.get("Theil U2"), 0.02),
]
_t = pd.DataFrame(_baris, columns=["Ukuran", "EViews", "Colab", "Toleransi"]).set_index("Ukuran")
_t["Selisih"] = _t["Colab"] - _t["EViews"]
_t["Status"] = np.where(_t["Selisih"].abs() <= _t["Toleransi"], "✅ SESUAI", "❌ BEDA")
display(_t)
print("Catatan: angka EViews dibulatkan; selisih kecil wajar karena cara inisialisasi likelihood berbeda.\n"
      "Theil U2 di sini dihitung di dalam periode uji sehingga bisa sedikit berbeda dari EViews.")

In [ ]:
#@title Langkah 10 — Unduh hasil
#@markdown Pilih file yang ingin diunduh.
LAPORAN_EXCEL = True #@param {type:"boolean"}
HASIL_CSV = True #@param {type:"boolean"}
GRAFIK_PNG = True #@param {type:"boolean"}
RINGKASAN_TXT = True #@param {type:"boolean"}

import re
from google.colab import files
_dasar = re.sub(r"[^A-Za-z0-9_-]+", "_", f"proyeksi_{KOLOM_TARGET}").strip("_")
_keluaran = {}
if LAPORAN_EXCEL: _keluaran[f"{_dasar}_laporan.xlsx"] = er.excel_laporan(hasil)
if HASIL_CSV: _keluaran[f"{_dasar}.csv"] = er.csv_hasil(hasil)
if GRAFIK_PNG: _keluaran[f"{_dasar}.png"] = vz.png_hasil(hasil, KOLOM_TARGET)
if RINGKASAN_TXT: _keluaran[f"{_dasar}_ringkasan.txt"] = er.teks_ringkasan(hasil)
(FOLDER / "hasil").mkdir(exist_ok=True)
for _nama, _isi in _keluaran.items():
    _path = FOLDER / "hasil" / _nama
    _path.write_bytes(_isi)
    files.download(str(_path))
    print("⬇️", _nama)

---
## Opsional

**Uji otomatis** memeriksa bahwa mesin bekerja benar di Colab (termasuk estimasi dengan statsmodels dan kesamaan dengan kode acuan).

**Tampilan web** menjalankan *ARIMA Forecasting Platform* dari Colab dengan tautan sementara.
⚠️ Tautan `...trycloudflare.com` bersifat **publik** selama berjalan: jangan dibagikan bila datanya rahasia.
Untuk menutupnya: **Runtime → Disconnect and delete runtime**.

In [ ]:
#@title Uji otomatis (opsional)
TES = {
    'tests/test_modules.py': '"""\nUji modul. Jalankan dari folder proyek:  python -m pytest -q\nTes yang membutuhkan statsmodels otomatis dilewati bila statsmodels belum terpasang.\n"""\nimport io\nimport sys\nimport types\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport pytest\n\nAKAR = Path(__file__).resolve().parent.parent\nsys.path.insert(0, str(AKAR))\n\nfrom modules import data_loader as dl  # noqa: E402\nfrom modules import data_preprocessing as dp  # noqa: E402\nfrom modules import evaluation as ev  # noqa: E402\nfrom modules import exploration as ex  # noqa: E402\nfrom modules import forecasting as fc  # noqa: E402\nfrom modules.arima_model import SpesifikasiModel, nama_parameter, peringatan_model  # noqa: E402\n\nCONTOH_XLSX = AKAR / "sample_data" / "sample_dataset.xlsx"\nCONTOH_CSV = AKAR / "sample_data" / "sample_dataset.csv"\n\n\n# ------------------------------------------------------------------ 1-2. file valid\n@pytest.mark.parametrize("path", [CONTOH_XLSX, CONTOH_CSV])\ndef test_file_contoh_terbaca(path):\n    df = dl.baca_file(str(path))\n    assert list(df.columns) == ["tanggal", "outflow"]\n    v = dp.siapkan_seri(df, *dl.tebak_kolom(df), "Bulanan")\n    assert v.ok and not v.peringatan\n    assert len(v.seri) == 66 and str(v.seri.index.freq) in ("<MonthEnd>", "M")\n\n\ndef test_csv_titik_koma_dan_format_indonesia():\n    teks = "periode;nilai\\n2021M01;1.234.567,5\\n2021M02;200.000\\n2021M03;300000\\n"\n    f = io.StringIO(teks)\n    f.name = "x.csv"\n    df = dl.baca_file(f)\n    v = dp.siapkan_seri(df, "periode", "nilai", "Bulanan")\n    assert v.seri.tolist() == [1234567.5, 200000.0, 300000.0]\n\n\ndef test_format_file_salah():\n    with pytest.raises(dl.GalatData):\n        dl.baca_file("data.txt")\n\n\n# ------------------------------------------------------------------ 3. nilai kosong\ndef _df(tanggal, nilai):\n    return pd.DataFrame({"t": tanggal, "v": nilai})\n\n\ndef test_periode_hilang_dilaporkan_tidak_dihapus():\n    per = [f"2021M{m:02d}" for m in range(1, 13) if m != 5]\n    v = dp.siapkan_seri(_df(per, np.arange(1.0, 12.0)), "t", "v", "Bulanan")\n    assert v.ok\n    assert len(v.seri) == 12 and np.isnan(v.seri.loc["2021-05"])\n    assert any("tidak ada di data" in p for p in v.peringatan)\n\n\ndef test_nilai_kosong_dilaporkan():\n    per = [f"2021M{m:02d}" for m in range(1, 13)]\n    nilai = [1.0] * 12\n    nilai[3] = None\n    v = dp.siapkan_seri(_df(per, nilai), "t", "v", "Bulanan")\n    assert v.ok and v.seri.isna().sum() == 1\n    assert any("kosong" in p for p in v.peringatan)\n\n\n# ------------------------------------------------------------------ 4. tanggal tidak valid\ndef test_tanggal_tidak_valid():\n    v = dp.siapkan_seri(_df(["abc", "def"] * 6, range(12)), "t", "v", "Bulanan")\n    assert not v.ok and "tidak bisa dibaca" in v.galat[0]\n\n\ndef test_periode_ganda_salah_frekuensi():\n    harian = pd.date_range("2021-01-01", periods=40, freq="D")\n    v = dp.siapkan_seri(_df(harian, range(40)), "t", "v", "Bulanan")\n    assert not v.ok and "periode ganda" in v.galat[0]\n\n\ndef test_tanggal_format_indonesia_hari_dulu():\n    v = dp.siapkan_seri(_df([f"01/{m:02d}/2021" for m in range(2, 13)], range(1, 12)), "t", "v", "Bulanan")\n    assert v.ok, v.galat\n    assert dp.label_periode(v.seri.index[0]) == "2021-02" and len(v.seri) == 11\n\n\n# ------------------------------------------------------------------ 5. target bukan angka\ndef test_target_bukan_angka():\n    per = [f"2021M{m:02d}" for m in range(1, 13)]\n    v = dp.siapkan_seri(_df(per, ["tinggi", "rendah"] * 6), "t", "v", "Bulanan")\n    assert not v.ok and "bukan angka" in v.galat[0]\n\n\ndef test_kolom_sama():\n    v = dp.siapkan_seri(_df(["2021M01"], [1]), "t", "t", "Bulanan")\n    assert not v.ok\n\n\n# ------------------------------------------------------------------ 6. observasi kurang\ndef test_observasi_terlalu_sedikit():\n    per = [f"2021M{m:02d}" for m in range(1, 6)]\n    v = dp.siapkan_seri(_df(per, range(1, 6)), "t", "v", "Bulanan")\n    assert not v.ok and "terlalu sedikit" in v.galat[0]\n\n\ndef test_minimal_observasi_model_musiman():\n    assert SpesifikasiModel(seasonal_order=(1, 0, 0, 12)).minimal_observasi() >= 18\n    assert SpesifikasiModel(order=(1, 0, 0), seasonal_order=(0, 0, 0, 12)).minimal_observasi() == 10\n\n\n# ------------------------------------------------------------------ 7. nol/negatif dengan log\ndef test_log_menolak_nol_negatif():\n    s = pd.Series([1.0, 0.0, -2.0, 4.0], index=pd.period_range("2021-01", periods=4, freq="M"))\n    assert dp.cek_transformasi(s, "log")\n    assert not dp.cek_transformasi(s, "tanpa")\n\n\ndef test_log_dan_kebalikannya():\n    s = pd.Series([100.0, 200.0, 50.0], index=pd.period_range("2021-01", periods=3, freq="M"))\n    for mode in dp.TRANSFORMASI:\n        balik = dp.ke_skala_asli(dp.ke_skala_model(s, mode), mode)\n        harapan = np.exp(s) if mode == "sudah_log" else s\n        np.testing.assert_allclose(balik, harapan)\n\n\ndef test_petunjuk_log_ganda():\n    s = pd.Series(np.linspace(11.5, 13.5, 24), index=pd.period_range("2021-01", periods=24, freq="M"))\n    assert dp.petunjuk_log_ganda(s, "log")\n    assert dp.petunjuk_log_ganda(s * 1e5, "log") is None\n\n\n# ------------------------------------------------------------------ frekuensi lain\n@pytest.mark.parametrize("frek,tanggal,n", [\n    ("Triwulanan", ["2020Q1", "2020Q2", "2020Q3", "2020Q4"] * 1 + ["2021Q1", "2021Q2", "2021Q3", "2021Q4",\n                                                                     "2022Q1", "2022Q2", "2022Q3"], 11),\n    ("Mingguan", pd.date_range("2024-01-01", periods=30, freq="W-MON"), 30),\n    ("Harian", pd.date_range("2024-01-01", periods=30, freq="D"), 30),\n])\ndef test_frekuensi_lain(frek, tanggal, n):\n    v = dp.siapkan_seri(_df(tanggal, np.arange(1.0, n + 1)), "t", "v", frek)\n    assert v.ok, v.galat\n    assert len(v.seri) == n\n    assert dp.label_periode(v.seri.index[0])\n\n\n# ------------------------------------------------------------------ evaluasi\ndef test_metrik_definisi():\n    a, f = [100, 200, 150], [110, 180, 150]\n    m = ev.ukuran_akurasi(a, f)\n    assert m["MAE"] == pytest.approx(10.0)\n    assert m["RMSE"] == pytest.approx(np.sqrt((100 + 400 + 0) / 3))\n    assert m["MAPE (%)"] == pytest.approx((10 / 100 + 20 / 200 + 0) / 3 * 100)\n    assert m["Theil U2"] == pytest.approx(0.1940285, rel=1e-6)\n\n\ndef test_metrik_sama_dengan_kode_acuan():\n    sys.path.insert(0, str(AKAR / "reference"))\n    import arima_engine as ref\n    rng = np.random.default_rng(1)\n    a, f = rng.uniform(50, 500, 12), rng.uniform(50, 500, 12)\n    baru, lama = ev.ukuran_akurasi(a, f), ref.ukuran_akurasi(a, f)\n    for k in lama:\n        assert baru[k] == pytest.approx(lama[k], rel=1e-12)\n\n\ndef test_mape_mengecualikan_nol():\n    m = ev.ukuran_akurasi([0, 100], [10, 110])\n    assert m["MAPE (%)"] == pytest.approx(10.0)\n    assert m["Aktual = 0 (dikecualikan dari MAPE)"] == 1\n    assert m["MAE"] == pytest.approx(10.0)\n\n\n# ------------------------------------------------------------------ baseline (sama dengan kode acuan)\ndef test_baseline_sama_dengan_kode_acuan():\n    sys.path.insert(0, str(AKAR / "reference"))\n    import arima_engine as ref\n    df = dl.baca_file(str(CONTOH_XLSX))\n    y = dp.siapkan_seri(df, "tanggal", "outflow").seri\n    latih, uji = y.loc[:"2025-06"], y.loc["2025-07":].index\n    np.testing.assert_allclose(fc.baseline_naif_musiman(latih, uji, 12), ref.baseline_naif_musiman(latih, uji))\n    baru = np.exp(fc.baseline_rata_musiman(np.log(latih), uji, 12))\n    np.testing.assert_allclose(baru, ref.baseline_rata_bulanan(latih, uji))\n\n\n# ------------------------------------------------------------------ peringatan model\ndef test_peringatan_model_degeneratif():\n    params = pd.Series({"C": 12.68, "ar.S.L12": 1.0, "ma.S.L12": -0.999895, "sigma2": 0.4})\n    bse = pd.Series({"C": 0.22, "ar.S.L12": 7.39e-5, "ma.S.L12": 9.3e-5, "sigma2": 0.09})\n    pesan = peringatan_model(params, bse, np.ones(12), np.ones(12) * 0.99999, 12)\n    assert len(pesan) == 4\n    params2 = pd.Series({"C": 12.68, "ar.S.L12": 0.448, "sigma2": 0.4})\n    assert peringatan_model(params2, params2 * 0 + 0.1, np.full(12, 0.448 ** (1 / 12)), np.array([]), 12) == []\n\n\ndef test_nama_parameter():\n    assert [nama_parameter(n) for n in ["ar.S.L12", "ma.L1", "sigma2", "C"]] == ["SAR(12)", "MA(1)", "SIGMASQ", "C"]\n\n\n# ------------------------------------------------------------------ eksplorasi & PNG\ndef test_statistik_deskriptif():\n    s = pd.Series([1.0, 2.0, None, 4.0], index=pd.period_range("2021-01", periods=4, freq="M"))\n    st = ex.statistik_deskriptif(s)\n    assert st["Observasi terisi"] == 3 and st["Median"] == 2.0\n    assert ex.daftar_kosong(s) == ["2021-03"]\n\n\ndef test_png_dari_hasil_tiruan():\n    from modules.visualization import png_hasil\n    idx = pd.period_range("2021-01", periods=24, freq="M")\n    y = pd.Series(np.arange(24.0) + 10, index=idx)\n    uji = idx[18:]\n    depan = pd.period_range("2023-01", periods=6, freq="M")\n    h = types.SimpleNamespace(\n        pengaturan=types.SimpleNamespace(awal_latih=idx[0], akhir_latih=idx[17], akhir_uji=idx[-1],\n                                         tingkat_kepercayaan=95, spek=SpesifikasiModel()),\n        y_asli=y, akhir_data_proyeksi=idx[-1],\n        backtest={"tabel": pd.DataFrame({"aktual": y[uji], "ramalan": y[uji] + 1, "batas_bawah": y[uji] - 2,\n                                         "batas_atas": y[uji] + 3}, index=uji)},\n        ramalan=pd.DataFrame({"ramalan": np.ones(6) * 30, "batas_bawah": np.ones(6) * 25,\n                              "batas_atas": np.ones(6) * 35}, index=depan),\n    )\n    png = png_hasil(h, "outflow")\n    assert png[:8] == b"\\x89PNG\\r\\n\\x1a\\n" and len(png) > 10_000\n\n\n# ------------------------------------------------------------------ 8-10. butuh statsmodels\n@pytest.fixture\ndef seri_sar12():\n    """Proses SAR(12) sintetis: log y = 12,6 + u, u_t = 0,5 u_{t-12} + e."""\n    rng = np.random.default_rng(42)\n    n = 240\n    u = np.zeros(n + 120)\n    e = rng.normal(0, 0.3, n + 120)\n    for t in range(12, n + 120):\n        u[t] = 0.5 * u[t - 12] + e[t]\n    idx = pd.period_range("2000-01", periods=n, freq="M")\n    return pd.Series(np.exp(12.6 + u[120:]), index=idx, name="outflow")\n\n\ndef _pengaturan(seri, horizon=12, spek=None, uji=True):\n    from modules.pipeline import Pengaturan\n    return Pengaturan(spek=spek or SpesifikasiModel(), transformasi="log", awal_latih=seri.index[0],\n                      akhir_latih=seri.index[-25], akhir_uji=seri.index[-1] if uji else None, horizon=horizon)\n\n\ndef test_estimasi_memulihkan_parameter(seri_sar12):\n    pytest.importorskip("statsmodels")\n    from modules.pipeline import jalankan\n    h = jalankan(seri_sar12, _pengaturan(seri_sar12))\n    k = h.koefisien_akhir["Koefisien"]\n    assert k["C"] == pytest.approx(12.6, abs=0.1)\n    assert k["SAR(12)"] == pytest.approx(0.5, abs=0.15)\n    assert h.akurasi is not None and len(h.backtest["tabel"]) == 24\n\n\n@pytest.mark.parametrize("horizon", [1, 12, 36])\ndef test_horizon(seri_sar12, horizon):\n    pytest.importorskip("statsmodels")\n    from modules.pipeline import jalankan\n    h = jalankan(seri_sar12, _pengaturan(seri_sar12, horizon, uji=False))\n    assert len(h.ramalan) == horizon and h.backtest is None\n    assert h.ramalan.index[0] == seri_sar12.index[-1] + 1\n    assert (h.ramalan["batas_bawah"] < h.ramalan["ramalan"]).all()\n    assert (h.ramalan["ramalan"] < h.ramalan["batas_atas"]).all()\n\n\ndef test_backtest_tanpa_kebocoran(seri_sar12):\n    """Mengubah nilai periode uji tidak boleh mengubah ramalan backtest."""\n    pytest.importorskip("statsmodels")\n    from modules.pipeline import jalankan\n    peng = _pengaturan(seri_sar12)\n    a = jalankan(seri_sar12, peng).backtest["tabel"]["ramalan"]\n    rusak = seri_sar12.copy()\n    rusak.iloc[-24:] *= 5\n    b = jalankan(rusak, peng).backtest["tabel"]["ramalan"]\n    np.testing.assert_allclose(a, b)\n\n\ndef test_sama_dengan_kode_acuan(seri_sar12):\n    pytest.importorskip("statsmodels")\n    sys.path.insert(0, str(AKAR / "reference"))\n    import arima_engine as ref\n    from modules.pipeline import jalankan\n    peng = _pengaturan(seri_sar12)\n    h = jalankan(seri_sar12, peng)\n    spek_ref = ref.SpesifikasiModel(order=(0, 0, 0), seasonal_order=(1, 0, 0, 12))\n    bt = ref.backtest(seri_sar12, spek_ref, str(peng.awal_latih), str(peng.akhir_latih), str(peng.akhir_uji))\n    np.testing.assert_allclose(h.backtest["tabel"]["ramalan"], bt["tabel"][spek_ref.label()], rtol=1e-8)\n\n\ndef test_model_manual_lain_dan_ekspor(seri_sar12):\n    pytest.importorskip("statsmodels")\n    from modules import export_results as exr\n    from modules.pipeline import jalankan, tabel_hasil\n    h = jalankan(seri_sar12, _pengaturan(seri_sar12, spek=SpesifikasiModel(order=(1, 0, 1),\n                                                                         seasonal_order=(1, 0, 0, 12))))\n    assert {"AR(1)", "MA(1)", "SAR(12)", "C", "SIGMASQ"} <= set(h.koefisien_akhir.index)\n    assert len(tabel_hasil(h)) == 24 + 12\n    xls = pd.read_excel(io.BytesIO(exr.excel_laporan(h)), sheet_name=None)\n    assert {"Pengaturan", "Hasil", "Evaluasi", "Koefisien", "Statistik model"} <= set(xls)\n    assert exr.csv_hasil(h).startswith("\ufeffPeriode".encode("utf-8"))\n    assert b"MODEL UNTUK PROYEKSI" in exr.teks_ringkasan(h)\n'
}
for _nama, _isi in TES.items():
    (FOLDER / _nama).parent.mkdir(parents=True, exist_ok=True)
    (FOLDER / _nama).write_text(_isi)
!cd /content/sarima_app && python -m pytest -q -p no:cacheprovider tests

In [ ]:
#@title Buka tampilan web (opsional)
BERKAS_WEB = {
    'app.py': '"""\nARIMA Forecasting Platform — titik masuk aplikasi.\nJalankan dari folder proyek:  python -m streamlit run app.py\n"""\nimport streamlit as st\n\nfrom halaman import komponen\n\nst.set_page_config(page_title="ARIMA Forecasting Platform", page_icon="📈", layout="wide")\nkomponen.gaya()\n\nnavigasi = st.navigation({\n    "Mulai": [\n        st.Page("halaman/beranda.py", title="Beranda", icon=":material/home:", default=True),\n    ],\n    "Alur kerja": [\n        st.Page("halaman/unggah.py", title="1. Unggah Data", icon=":material/upload_file:"),\n        st.Page("halaman/eksplorasi.py", title="2. Eksplorasi Data", icon=":material/query_stats:"),\n        st.Page("halaman/konfigurasi.py", title="3. Konfigurasi & Jalankan", icon=":material/tune:"),\n        st.Page("halaman/hasil.py", title="4. Hasil Proyeksi", icon=":material/show_chart:"),\n        st.Page("halaman/evaluasi.py", title="5. Evaluasi Model", icon=":material/fact_check:"),\n        st.Page("halaman/unduh.py", title="6. Unduh Hasil", icon=":material/download:"),\n    ],\n})\nkomponen.status_sidebar()\nnavigasi.run()\n',
    'halaman/__init__.py': '',
    'halaman/komponen.py': '"""Komponen tampilan bersama dan pengelolaan session state."""\nfrom __future__ import annotations\n\nimport streamlit as st\n\nfrom modules.data_preprocessing import label_periode\n\n# Kunci session state\nDF = "df_mentah"\nNAMA_FILE = "nama_file"\nSERI = "seri"\nFREKUENSI = "frekuensi"\nTARGET = "kolom_target"\nHASIL = "hasil"\n\n\ndef gaya():\n    st.markdown(\n        """\n        <style>\n        .block-container {padding-top: 2rem; max-width: 1300px;}\n        div[data-testid="stMetricValue"] {font-size: 1.5rem;}\n        .langkah {border: 1px solid rgba(128,128,128,.25); border-radius: 10px; padding: 14px 16px; height: 100%;}\n        .langkah b {color: #2563eb;}\n        </style>\n        """,\n        unsafe_allow_html=True,\n    )\n\n\ndef reset_hasil():\n    st.session_state.pop(HASIL, None)\n\n\ndef reset_data():\n    for k in (SERI, HASIL):\n        st.session_state.pop(k, None)\n\n\ndef status_sidebar():\n    with st.sidebar:\n        st.markdown("### Status proyek")\n        nama = st.session_state.get(NAMA_FILE)\n        seri = st.session_state.get(SERI)\n        hasil = st.session_state.get(HASIL)\n        st.markdown(f"{\'✅\' if nama else \'⬜\'} Data: {nama or \'belum diunggah\'}")\n        if seri is not None:\n            st.caption(f"{st.session_state.get(TARGET)} · {st.session_state.get(FREKUENSI)} · "\n                       f"{label_periode(seri.index.min())} – {label_periode(seri.index.max())}")\n        st.markdown(f"{\'✅\' if hasil else \'⬜\'} Model: {hasil.pengaturan.spek.label() if hasil else \'belum dijalankan\'}")\n\n\ndef perlu_seri():\n    """Hentikan halaman bila data belum siap."""\n    if st.session_state.get(SERI) is None:\n        st.info("Belum ada data yang siap dipakai. Mulai dari halaman **Unggah Data**.")\n        st.page_link("halaman/unggah.py", label="Ke halaman Unggah Data", icon=":material/upload_file:")\n        st.stop()\n    return st.session_state[SERI]\n\n\ndef perlu_hasil():\n    if st.session_state.get(HASIL) is None:\n        st.info("Model belum dijalankan. Atur model lalu klik **Jalankan Proyeksi** di halaman Konfigurasi.")\n        st.page_link("halaman/konfigurasi.py", label="Ke halaman Konfigurasi", icon=":material/tune:")\n        st.stop()\n    return st.session_state[HASIL]\n\n\ndef fmt_angka(v, desimal: int = 2) -> str:\n    """Format angka gaya Indonesia: 1.234.567,89"""\n    try:\n        teks = f"{float(v):,.{desimal}f}"\n    except (TypeError, ValueError):\n        return str(v)\n    return teks.replace(",", "_").replace(".", ",").replace("_", ".")\n',
    'halaman/beranda.py': 'import streamlit as st\n\nst.title("📈 ARIMA Forecasting Platform")\nst.markdown("##### Platform interaktif untuk proyeksi data deret waktu")\n\nst.markdown(\n    """\nPlatform ini membantu Anda membuat **proyeksi (forecast)** dari data deret waktu, misalnya\noutflow bulanan, penjualan, atau harga, dengan model **ARIMA**, tanpa perlu menulis kode.\n"""\n)\n\nwith st.expander("Apa itu ARIMA?", expanded=False):\n    st.markdown(\n        """\n**ARIMA** (*AutoRegressive Integrated Moving Average*) adalah model statistik yang meramal nilai\nke depan berdasarkan pola data itu sendiri:\n\n- **AR (p)**: nilai sekarang dipengaruhi nilai periode sebelumnya.\n- **I (d)**: berapa kali data dibedakan (*differencing*) agar stabil (stasioner).\n- **MA (q)**: nilai sekarang dipengaruhi kesalahan ramalan periode sebelumnya.\n\nBila ada pola yang berulang tiap tahun (musiman), dipakai **SARIMA** dengan komponen musiman\n(P, D, Q) dan panjang musim *s* (mis. 12 untuk data bulanan).\n"""\n    )\n\nst.markdown("### Alur kerja")\nlangkah = [\n    ("1", "Unggah data", "File Excel/CSV berisi kolom tanggal dan nilai."),\n    ("2", "Eksplorasi", "Lihat grafik, statistik, dan nilai kosong."),\n    ("3", "Konfigurasi", "Pilih transformasi, periode latih/uji, dan model."),\n    ("4", "Hasil", "Koefisien model, grafik, dan tabel ramalan."),\n    ("5", "Evaluasi", "MAE, RMSE, MAPE pada periode uji."),\n    ("6", "Unduh", "Excel, CSV, PNG, dan ringkasan teks."),\n]\nkolom = st.columns(len(langkah))\nfor k, (no, judul, isi) in zip(kolom, langkah):\n    k.markdown(f"<div class=\'langkah\'><b>{no}. {judul}</b><br><small>{isi}</small></div>", unsafe_allow_html=True)\n\nst.markdown("### Cara memakai")\nst.markdown(\n    """\n1. Buka **Unggah Data**, unggah file Anda (atau pakai data contoh), lalu pilih kolom tanggal, kolom yang ingin diramal, dan frekuensinya.\n2. Periksa data di **Eksplorasi Data**.\n3. Di **Konfigurasi & Jalankan**, atur model lalu klik **Jalankan Proyeksi**.\n4. Lihat hasilnya di **Hasil Proyeksi** dan **Evaluasi Model**, lalu unduh di **Unduh Hasil**.\n\nData yang Anda unggah hanya diproses selama sesi berjalan dan tidak disimpan oleh aplikasi.\n"""\n)\n\nif st.button("Mulai proyek baru", type="primary", icon=":material/arrow_forward:"):\n    st.switch_page("halaman/unggah.py")\n',
    'halaman/unggah.py': 'from pathlib import Path\n\nimport streamlit as st\n\nfrom halaman import komponen as kp\nfrom modules.data_loader import GalatData, baca_file, ringkasan_dataset, tebak_kolom\nfrom modules.data_preprocessing import FREKUENSI, label_periode, siapkan_seri\n\nCONTOH = Path(__file__).resolve().parent.parent / "sample_data" / "sample_dataset.xlsx"\n\nst.title("1. Unggah Data")\nst.caption("Unggah file Excel (.xlsx) atau CSV (.csv). Satu baris = satu periode.")\n\nc1, c2 = st.columns([3, 1])\nunggahan = c1.file_uploader("Pilih file", type=["xlsx", "xls", "csv"])\nc2.write("")\nc2.write("")\npakai_contoh = c2.button("Pakai data contoh", help="Data sintetis bulanan untuk mencoba aplikasi.")\n\nid_unggahan = getattr(unggahan, "file_id", None) or (unggahan.name if unggahan else None)\nif unggahan is not None and id_unggahan != st.session_state.get("id_unggahan"):\n    st.session_state["id_unggahan"] = id_unggahan  # hanya dibaca sekali per file baru\n    try:\n        st.session_state[kp.DF] = baca_file(unggahan)\n        st.session_state[kp.NAMA_FILE] = unggahan.name\n        kp.reset_data()\n    except GalatData as e:\n        st.error(str(e))\n        st.stop()\nelif pakai_contoh:\n    st.session_state[kp.DF] = baca_file(CONTOH)\n    st.session_state[kp.NAMA_FILE] = "sample_dataset.xlsx (contoh sintetis)"\n    kp.reset_data()\n\ndf = st.session_state.get(kp.DF)\nif df is None:\n    st.info("Belum ada file. Unggah file Anda atau klik **Pakai data contoh**.")\n    st.stop()\n\nst.success(f"File: **{st.session_state[kp.NAMA_FILE]}**")\nringkas = ringkasan_dataset(df)\nm1, m2, m3 = st.columns(3)\nm1.metric("Jumlah baris (observasi)", ringkas["jumlah_baris"])\nm2.metric("Jumlah kolom (variabel)", ringkas["jumlah_kolom"])\nm3.metric("Total sel kosong", int(df.isna().sum().sum()))\n\nt1, t2 = st.tabs(["Pratinjau data", "Informasi kolom"])\nt1.dataframe(df.head(50))\nt2.dataframe(ringkas["info_kolom"])\n\nst.subheader("Pilih kolom dan frekuensi")\ntebak_tgl, tebak_target = tebak_kolom(df)\nkolom = list(df.columns)\nc1, c2, c3 = st.columns(3)\nkolom_tgl = c1.selectbox("Kolom tanggal/periode", kolom, index=kolom.index(tebak_tgl),\n                         help="Format yang dikenali: 2021-01-31, 31/01/2021, tanggal Excel, 2021M01, 2021Q1.")\nkolom_target = c2.selectbox("Variabel yang diramal (target)", kolom, index=kolom.index(tebak_target),\n                            help="Kolom angka yang ingin diproyeksikan, dalam satuan aslinya.")\nfrekuensi = c3.selectbox("Frekuensi data", list(FREKUENSI), index=0,\n                         help="Seberapa sering data dicatat. Menentukan panjang musim bawaan.")\n\nvalidasi = siapkan_seri(df, kolom_tgl, kolom_target, frekuensi)\nfor p in validasi.peringatan:\n    st.warning(p)\nif not validasi.ok:\n    for g in validasi.galat:\n        st.error(g)\n    st.session_state.pop(kp.SERI, None)\n    st.stop()\n\nseri = validasi.seri\nlama = st.session_state.get(kp.SERI)\nif lama is None or not lama.equals(seri) or st.session_state.get(kp.TARGET) != kolom_target:\n    kp.reset_hasil()\nst.session_state[kp.SERI] = seri\nst.session_state[kp.FREKUENSI] = frekuensi\nst.session_state[kp.TARGET] = kolom_target\n\nm1, m2, m3 = st.columns(3)\nm1.metric("Rentang periode", f"{label_periode(seri.index.min())} – {label_periode(seri.index.max())}")\nm2.metric("Jumlah periode", len(seri))\nm3.metric("Periode kosong", int(seri.isna().sum()))\nst.success("Data siap. Lanjutkan ke eksplorasi atau langsung ke konfigurasi model.")\nc1, c2 = st.columns(2)\nif c1.button("Eksplorasi data", icon=":material/query_stats:"):\n    st.switch_page("halaman/eksplorasi.py")\nif c2.button("Konfigurasi model", type="primary", icon=":material/tune:"):\n    st.switch_page("halaman/konfigurasi.py")\n',
    'halaman/eksplorasi.py': 'import streamlit as st\n\nfrom halaman import komponen as kp\nfrom modules.data_preprocessing import FREKUENSI\nfrom modules.exploration import daftar_kosong, dekomposisi, statistik_deskriptif\nfrom modules.visualization import grafik_seri\n\nseri = kp.perlu_seri()\ntarget = st.session_state[kp.TARGET]\n\nst.title("2. Eksplorasi Data")\nst.plotly_chart(grafik_seri(seri, f"{target} ({st.session_state[kp.FREKUENSI].lower()})", target))\n\nc1, c2 = st.columns([1, 1])\nwith c1:\n    st.subheader("Statistik deskriptif")\n    stat = statistik_deskriptif(seri)\n    tampil = {k: (kp.fmt_angka(v) if isinstance(v, float) else v) for k, v in stat.items()}\n    st.dataframe({"Ukuran": list(tampil), "Nilai": [str(v) for v in tampil.values()]}, hide_index=True)\nwith c2:\n    st.subheader("Nilai kosong")\n    kosong = daftar_kosong(seri)\n    if kosong:\n        st.warning(f"{len(kosong)} periode kosong: {\', \'.join(kosong)}. "\n                   "Periode ini tidak diisi; model memperlakukannya sebagai data hilang.")\n    else:\n        st.success("Tidak ada periode kosong.")\n\nst.subheader("Dekomposisi (opsional)")\nst.caption("Memisahkan data menjadi tren, pola musiman, dan sisa. Hanya untuk informasi; tidak memengaruhi model.")\nwith st.expander("Tampilkan dekomposisi"):\n    s_bawaan = FREKUENSI[st.session_state[kp.FREKUENSI]][1]\n    c1, c2 = st.columns(2)\n    s = c1.number_input("Panjang musim", 2, 365, s_bawaan)\n    jenis = c2.radio("Jenis", ["additive", "multiplicative"], horizontal=True,\n                     format_func=lambda x: "Aditif" if x == "additive" else "Multiplikatif")\n    try:\n        d = dekomposisi(seri, int(s), jenis)\n        import plotly.graph_objects as go\n        from plotly.subplots import make_subplots\n        fig = make_subplots(rows=4, cols=1, shared_xaxes=True, subplot_titles=list(d.columns))\n        for i, kol in enumerate(d.columns, start=1):\n            fig.add_trace(go.Scatter(x=d.index, y=d[kol], name=kol, showlegend=False), row=i, col=1)\n        fig.update_layout(height=650, template="plotly_white", margin=dict(t=40, b=10))\n        st.plotly_chart(fig)\n    except ValueError as e:\n        st.info(str(e))\n',
    'halaman/konfigurasi.py': 'import traceback\n\nimport streamlit as st\n\nfrom halaman import komponen as kp\nfrom modules.arima_model import SpesifikasiModel\nfrom modules.data_preprocessing import FREKUENSI, TRANSFORMASI, cek_transformasi, label_periode, petunjuk_log_ganda\nfrom modules.pipeline import GalatPengaturan, Pengaturan, jalankan\n\nseri = kp.perlu_seri()\ns_bawaan = FREKUENSI[st.session_state[kp.FREKUENSI]][1]\nperiode = list(seri.dropna().index)\nn = len(periode)\n\nst.title("3. Konfigurasi & Jalankan")\n\n# ---------------------------------------------------------------- transformasi\nst.subheader("Transformasi data")\ntransformasi = st.radio(\n    "Bagaimana data diolah sebelum dimodelkan?", list(TRANSFORMASI), format_func=TRANSFORMASI.get,\n    help="Log membuat pola musiman yang membesar-mengecil menjadi lebih stabil. Hasil ramalan selalu "\n         "dikembalikan ke skala asli (dengan exp) secara otomatis.")\nif (petunjuk := petunjuk_log_ganda(seri, transformasi)):\n    st.warning(petunjuk)\nfor g in cek_transformasi(seri, transformasi):\n    st.error(g)\nif transformasi == "sudah_log":\n    st.info("Data dianggap sudah berupa log. Model memakai data apa adanya; hasil ditampilkan dalam skala asli (exp).")\n\n# ---------------------------------------------------------------- periode\nst.subheader("Periode latih dan uji")\nst.caption("Data dibagi menurut urutan waktu (tidak diacak). Model hanya belajar dari data latih; "\n           "data uji dipakai untuk mengukur akurasi ramalan.")\npakai_uji = st.checkbox("Sisihkan periode uji (disarankan)", value=n > 2 * s_bawaan + 10)\npanjang_uji = s_bawaan if n > 3 * s_bawaan else max(1, n // 5)\nc1, c2, c3 = st.columns(3)\nawal_latih = c1.selectbox("Awal data latih", periode, index=0, format_func=label_periode)\nopsi_akhir = [p for p in periode if p > awal_latih]\nidx_akhir = max(0, len(opsi_akhir) - 1 - (panjang_uji if pakai_uji else 0))\nakhir_latih = c2.selectbox("Akhir data latih", opsi_akhir, index=idx_akhir, format_func=label_periode)\nakhir_uji = None\nif pakai_uji:\n    opsi_uji = [p for p in periode if p > akhir_latih]\n    if opsi_uji:\n        akhir_uji = c3.selectbox("Akhir data uji", opsi_uji, index=len(opsi_uji) - 1, format_func=label_periode)\n    else:\n        c3.warning("Tidak ada data setelah akhir data latih.")\n\n# ---------------------------------------------------------------- model\nst.subheader("Spesifikasi model")\nst.info("**Spesifikasi manual.** Pemilihan model otomatis (auto-ARIMA) belum tersedia karena tidak ada di kode "\n        "acuan; orde diisi dari hasil identifikasi Anda (mis. EViews). Nilai awal: (0,0,0)(1,0,0) — model SAR musiman.")\nwith st.expander("Penjelasan parameter"):\n    st.markdown(\n        """\n| Parameter | Arti |\n|---|---|\n| **p** | Jumlah lag AR: pengaruh nilai periode-periode sebelumnya |\n| **d** | Berapa kali data dibedakan (differencing). 0 bila data sudah stasioner |\n| **q** | Jumlah lag MA: pengaruh kesalahan ramalan sebelumnya |\n| **P, D, Q** | Sama seperti p, d, q tetapi pada lag musiman (mis. 12 bulan lalu) |\n| **s** | Panjang satu musim: 12 bulanan, 4 triwulanan, 52 mingguan, 7 harian |\n| **Konstanta C** | Rata-rata deret (seperti C di EViews) |\n"""\n    )\nc = st.columns(7)\np = c[0].number_input("p", 0, 5, 0)\nd = c[1].number_input("d", 0, 2, 0)\nq = c[2].number_input("q", 0, 5, 0)\nP = c[3].number_input("P", 0, 3, 1)\nD = c[4].number_input("D", 0, 1, 0)\nQ = c[5].number_input("Q", 0, 3, 0)\ns = c[6].number_input("s", 1, 366, s_bawaan)\nkonstanta = st.checkbox("Sertakan konstanta C (rata-rata)", value=True)\nif konstanta and (d or D):\n    st.warning("Dengan differencing (d atau D > 0), konstanta berarti tren/drift. Pastikan ini memang dimaksud.")\n\nwith st.expander("Opsi lanjutan"):\n    paksa_stasioner = st.checkbox("Paksa model stasioner", value=False,\n                                  help="EViews tidak memaksa. Aktifkan bila estimasi lari ke koefisien AR = 1.")\n    paksa_invertible = st.checkbox("Paksa model invertible", value=False)\n    koreksi_bias = st.checkbox("Koreksi bias saat mengembalikan dari log", value=False,\n                               help="exp(ramalan log) adalah median, bukan rata-rata. Koreksi mengalikan "\n                                    "dengan exp(σ²/2). Kode acuan/EViews tidak memakainya.")\n\n# ---------------------------------------------------------------- proyeksi\nst.subheader("Proyeksi")\nc1, c2 = st.columns(2)\nhorizon = c1.number_input("Horizon (jumlah periode ke depan)", 1, 5 * s_bawaan if s_bawaan > 1 else 60, s_bawaan)\ntingkat = c2.select_slider("Tingkat kepercayaan selang ramalan", [80, 90, 95, 99], value=95)\n\nspek = SpesifikasiModel(order=(int(p), int(d), int(q)), seasonal_order=(int(P), int(D), int(Q), int(s)),\n                        konstanta=konstanta, paksa_stasioner=paksa_stasioner, paksa_invertible=paksa_invertible)\npeng = Pengaturan(spek=spek, transformasi=transformasi, awal_latih=awal_latih, akhir_latih=akhir_latih,\n                  akhir_uji=akhir_uji, horizon=int(horizon), tingkat_kepercayaan=int(tingkat),\n                  koreksi_bias=koreksi_bias)\n\nst.divider()\nst.markdown(f"**Model:** {spek.label()} · **Latih:** {label_periode(awal_latih)}–{label_periode(akhir_latih)}"\n            + (f" · **Uji:** {label_periode(akhir_latih + 1)}–{label_periode(akhir_uji)}" if akhir_uji else " · tanpa uji")\n            + f" · **Horizon:** {horizon}")\n\nif st.button("Jalankan Proyeksi", type="primary", icon=":material/play_arrow:"):\n    with st.status("Menjalankan model...", expanded=True) as status:\n        try:\n            st.write("Memeriksa data dan pengaturan, mengestimasi model, membuat ramalan...")\n            hasil = jalankan(seri, peng)\n            st.session_state[kp.HASIL] = hasil\n            status.update(label="Selesai", state="complete")\n        except GalatPengaturan as e:\n            status.update(label="Pengaturan perlu diperbaiki", state="error")\n            for baris in str(e).splitlines():\n                st.error(baris)\n        except Exception as e:\n            status.update(label="Estimasi gagal", state="error")\n            st.error("Model tidak berhasil diestimasi dengan pengaturan ini. Coba model yang lebih sederhana, "\n                     "periksa nilai kosong, atau aktifkan opsi \'Paksa model stasioner\'.")\n            with st.expander("Detail teknis (untuk pengembang)"):\n                st.code(f"{type(e).__name__}: {e}\\n\\n{traceback.format_exc()}")\n\nif st.session_state.get(kp.HASIL) is not None:\n    st.success(f"Hasil tersedia untuk {st.session_state[kp.HASIL].pengaturan.spek.label()}.")\n    if st.button("Lihat hasil", icon=":material/show_chart:"):\n        st.switch_page("halaman/hasil.py")\n',
    'halaman/hasil.py': 'import streamlit as st\n\nfrom halaman import komponen as kp\nfrom modules.arima_model import tabel_koefisien\nfrom modules.data_preprocessing import label_periode\nfrom modules.pipeline import tabel_hasil\nfrom modules.visualization import grafik_acf, grafik_hasil\n\nh = kp.perlu_hasil()\npeng = h.pengaturan\ntarget = st.session_state.get(kp.TARGET, "Nilai")\n\nst.title("4. Hasil Proyeksi")\nfor p in h.diagnostik_akhir["peringatan"]:\n    st.warning(p)\nfor p in h.peringatan:\n    st.info(p)\n\nm = st.columns(4)\nm[0].metric("Model", peng.spek.label())\nm[1].metric("Data latih", f"{label_periode(peng.awal_latih)} – {label_periode(peng.akhir_latih)}")\nm[2].metric("Periode uji", f"{label_periode(peng.akhir_latih + 1)} – {label_periode(peng.akhir_uji)}"\n            if peng.akhir_uji else "Tidak ada")\nm[3].metric("Horizon proyeksi", f"{peng.horizon} periode")\n\nst.plotly_chart(grafik_hasil(h, target))\n\nst.subheader("Tabel aktual dan ramalan")\nst.caption("Baris \'Uji (backtest)\': ramalan dari model yang hanya memakai data latih. "\n           "Baris \'Proyeksi ke depan\': aktual belum ada. Semua nilai dalam skala asli data.")\ntabel = tabel_hasil(h)\nst.dataframe(tabel.style.format({c: kp.fmt_angka for c in tabel.columns if c != "Jenis"}, na_rep="—"))\n\nst.subheader("Estimasi model")\ntab_akhir, tab_bt, tab_diag, tab_ringkas = st.tabs(\n    ["Model untuk proyeksi", "Model backtest", "Diagnostik residual", "Ringkasan statsmodels"])\n\nwith tab_akhir:\n    st.caption(f"Diestimasi dengan data {label_periode(peng.awal_latih)} s.d. {label_periode(h.akhir_data_proyeksi)}.")\n    st.dataframe(h.koefisien_akhir.style.format("{:.6f}"))\n    st.dataframe(h.statistik_akhir.to_frame("Nilai").style.format("{:.6f}"))\n    st.caption("AIC/BIC statsmodels berupa nilai total; versi \'per obs.\' sebanding dengan angka EViews.")\n\nwith tab_bt:\n    if h.backtest is None:\n        st.info("Tidak ada periode uji.")\n    else:\n        st.caption(f"Diestimasi hanya dengan data latih {label_periode(peng.awal_latih)} s.d. "\n                   f"{label_periode(peng.akhir_latih)}; dipakai untuk ramalan periode uji.")\n        st.dataframe(tabel_koefisien(h.backtest["hasil"]).style.format("{:.6f}"))\n\nwith tab_diag:\n    diag = h.diagnostik_akhir\n    c1, c2 = st.columns(2)\n    with c1:\n        st.markdown("**Ljung-Box**: p > 0,05 berarti tidak ada autokorelasi residual yang tersisa.")\n        st.dataframe(diag["ljung_box"].rename(columns={"lb_stat": "Statistik Q", "lb_pvalue": "p-value"}))\n        jb = diag["jarque_bera"]\n        st.markdown(f"**Jarque-Bera**: {jb[\'statistik\']:.3f}, p = {jb[\'p_value\']:.4f} "\n                    f"→ residual {\'normal\' if jb[\'p_value\'] > 0.05 else \'tidak normal\'}.")\n        ar, ma = diag["akar_ar_invers_maks"], diag["akar_ma_invers_maks"]\n        st.markdown(f"**Akar invers maksimum** (harus < 1): AR = {ar:.4f}" if ar is not None else "AR: –")\n        if ma is not None:\n            st.markdown(f"MA = {ma:.4f}")\n    with c2:\n        st.plotly_chart(grafik_acf(diag))\n\nwith tab_ringkas:\n    st.code(str(h.hasil_akhir.summary()), language="text")\n',
    'halaman/evaluasi.py': 'import streamlit as st\n\nfrom halaman import komponen as kp\nfrom modules.data_preprocessing import label_periode\nfrom modules.evaluation import PENJELASAN\n\nh = kp.perlu_hasil()\npeng = h.pengaturan\n\nst.title("5. Evaluasi Model")\nif h.akurasi is None:\n    st.info("Tidak ada periode uji, jadi akurasi ramalan tidak bisa diukur. Aktifkan "\n            "\'Sisihkan periode uji\' di halaman Konfigurasi.")\n    st.stop()\n\nst.caption(f"Akurasi **di luar sampel**: model diestimasi dengan data {label_periode(peng.awal_latih)}–"\n           f"{label_periode(peng.akhir_latih)}, lalu meramal {label_periode(peng.akhir_latih + 1)}–"\n           f"{label_periode(peng.akhir_uji)} tanpa melihat nilai aktualnya.")\n\nutama = h.akurasi.loc[peng.spek.label()]\nm = st.columns(5)\nm[0].metric("MAE", kp.fmt_angka(utama["MAE"]))\nm[1].metric("RMSE", kp.fmt_angka(utama["RMSE"]))\nm[2].metric("MAPE", f"{kp.fmt_angka(utama[\'MAPE (%)\'])}%")\nm[3].metric("sMAPE", f"{kp.fmt_angka(utama[\'sMAPE (%)\'])}%")\nm[4].metric("Theil U2", kp.fmt_angka(utama.get("Theil U2"), 3))\n\nnol = int(utama.get("Aktual = 0 (dikecualikan dari MAPE)", 0))\nif nol:\n    st.warning(f"{nol} periode uji bernilai aktual 0 dan dikecualikan dari MAPE (pembagian dengan nol). "\n               "MAE, RMSE, dan sMAPE tetap menghitungnya.")\n\nst.subheader("Dibandingkan dengan ramalan sederhana")\nst.caption("**Naif musiman**: nilai periode yang sama satu musim sebelumnya. **Rata-rata musiman**: rata-rata "\n           "tiap posisi musim (mis. tiap bulan) di data latih. Model yang baik seharusnya lebih akurat dari keduanya.")\nst.dataframe(h.akurasi.style.format(kp.fmt_angka, subset=[c for c in h.akurasi.columns\n                                                           if c not in ("n", "Aktual = 0 (dikecualikan dari MAPE)")]))\n\nst.subheader("Arti setiap ukuran")\nfor nama, isi in PENJELASAN.items():\n    st.markdown(f"- **{nama}**: {isi}")\nst.caption("Periode uji yang pendek membuat ukuran-ukuran ini sensitif terhadap satu-dua periode yang meleset jauh. "\n           "Lihat juga kolom galat per periode di halaman Hasil.")\n',
    'halaman/unduh.py': 'import re\n\nimport streamlit as st\n\nfrom halaman import komponen as kp\nfrom modules import export_results as ex\nfrom modules.visualization import png_hasil\n\nh = kp.perlu_hasil()\ntarget = st.session_state.get(kp.TARGET, "Nilai")\ndasar = re.sub(r"[^A-Za-z0-9_-]+", "_", f"proyeksi_{target}").strip("_")\n\nst.title("6. Unduh Hasil")\nst.caption(f"Semua file dibuat dari model yang terakhir dijalankan: **{h.pengaturan.spek.label()}**.")\n\nc1, c2 = st.columns(2)\nwith c1:\n    st.markdown("**Laporan lengkap (Excel)**  \\nPengaturan, hasil, evaluasi, koefisien, statistik model.")\n    st.download_button("Unduh laporan .xlsx", ex.excel_laporan(h), f"{dasar}_laporan.xlsx",\n                       "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",\n                       type="primary", icon=":material/table_view:")\n    st.markdown("**Hasil ramalan (Excel)**")\n    st.download_button("Unduh hasil .xlsx", ex.excel_hasil_saja(h), f"{dasar}.xlsx",\n                       "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet", icon=":material/grid_on:")\n    st.markdown("**Hasil ramalan (CSV)**")\n    st.download_button("Unduh hasil .csv", ex.csv_hasil(h), f"{dasar}.csv", "text/csv", icon=":material/description:")\nwith c2:\n    st.markdown("**Grafik (PNG)**")\n    png = png_hasil(h, target)\n    st.image(png)\n    st.download_button("Unduh grafik .png", png, f"{dasar}.png", "image/png", icon=":material/image:")\n    st.markdown("**Ringkasan model (TXT)**")\n    st.download_button("Unduh ringkasan .txt", ex.teks_ringkasan(h), f"{dasar}_ringkasan.txt", "text/plain",\n                       icon=":material/article:")\n',
    '.streamlit/config.toml': '[theme]\nprimaryColor = "#2563eb"\nbackgroundColor = "#ffffff"\nsecondaryBackgroundColor = "#f3f4f6"\ntextColor = "#111827"\nfont = "sans serif"\n\n[server]\nmaxUploadSize = 50\n\n[browser]\ngatherUsageStats = false\n',
    'requirements.txt': 'streamlit>=1.40\npandas>=2.1\nnumpy>=1.26\nscipy>=1.11\nstatsmodels>=0.14.2\nplotly>=5.20\nmatplotlib>=3.8\nopenpyxl>=3.1\n'
}
for _nama, _isi in BERKAS_WEB.items():
    (FOLDER / _nama).parent.mkdir(parents=True, exist_ok=True)
    (FOLDER / _nama).write_text(_isi)

!pip -q install streamlit
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared
!cd /content/sarima_app && nohup python -m streamlit run app.py --server.port 8501 --server.headless true &> /content/streamlit.log &
!nohup /content/cloudflared tunnel --url http://localhost:8501 &> /content/cloudflared.log &
import time, re as _re
for _ in range(30):
    time.sleep(2)
    _m = _re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/cloudflared.log").read())
    if _m:
        print("🌐 Buka aplikasi di:", _m.group(0))
        break
else:
    print("Tautan belum muncul. Tunggu sebentar lalu jalankan ulang sel ini, atau lihat /content/cloudflared.log")